In [ ]:
# Cell 1 — Objective: Install only the required packages for this notebook; import nothing sensitive here.

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets ipyfilechooser paramiko tqdm numpy pandas locust requests

print("Cell 1 complete: Python environment ready (Single/Multi IPv6-frontend variant).")
print("NEXT: Run Cell 2 to load central variables (adds Topology selector) and validate OCI config.")


In [ ]:
# Cell 2 — Objective: Central variables and credentials (single source of truth; no network calls)
# Supports Single or Multi LB topology with IPv6 frontend → IPv4 backends. No secrets are printed.

import os
from datetime import datetime, timezone
import oci

# ===== OCI config (no secrets hardcoded; load from standard ~/.oci/config) =====
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg["tenancy"])  # used later by UI in Cell 3
USER_OCID    = os.environ.get("USER_OCID",    _cfg["user"])     # used later by UI in Cell 3
FINGERPRINT  = os.environ.get("FINGERPRINT",  _cfg["fingerprint"])  # used later by UI in Cell 3
OCI_PRIVATE_KEY_PATH = os.path.expanduser(os.environ.get("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", "")))
PRIVATE_KEY_PASSPHRASE = os.environ.get("OCI_PASSPHRASE", os.environ.get("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")))
REGION = os.environ.get("REGION", _cfg["region"])  # will be overridable in Cell 3

# ===== SSH keys defaults (can be changed in Cell 3) =====
_default_ssh_pub  = os.path.expanduser(os.environ.get("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub"))
_default_ssh_priv = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa"))
SSH_PUBLIC_KEY_PATH = _default_ssh_pub  if os.path.exists(_default_ssh_pub)  else ""
SSH_PRIVATE_KEY_PATH = _default_ssh_priv if os.path.exists(_default_ssh_priv) else ""

# ===== TLS OFFLOAD at the LB: browse PEMs or generate self‑signed in Cell 3 =====
# Mode: "browse" (default) or "generate"
LB_CERT_MODE = os.environ.get("LB_CERT_MODE", "browse").strip().lower()  # "browse" | "generate"

# If browsing existing PEMs (these will be set via Cell 3 pickers or environment)
LB_CERT_PEM_PATH = os.path.expanduser(os.environ.get("LB_CERT_PEM_PATH", ""))
LB_KEY_PEM_PATH  = os.path.expanduser(os.environ.get("LB_KEY_PEM_PATH",  ""))
LB_CA_PEM_PATH   = os.path.expanduser(os.environ.get("LB_CA_PEM_PATH",   ""))  # optional

# If generating PEMs (defaults suitable for high‑CPS)
LB_CERT_KIND   = os.path.expanduser(os.environ.get("LB_CERT_KIND", "ecdsa")).strip().lower()  # "ecdsa" | "rsa"
LB_ECDSA_CURVE = os.environ.get("LB_ECDSA_CURVE", "secp256r1").strip()  # P‑256
LB_RSA_BITS    = int(os.environ.get("LB_RSA_BITS", "2048"))
LB_CERT_CN     = os.environ.get("LB_CERT_CN", "lb.local").strip()
LB_CERT_DAYS   = int(os.environ.get("LB_CERT_DAYS", "3650"))
LB_PEM_OUTPUT_DIR = os.path.expanduser(os.environ.get("LB_PEM_OUTPUT_DIR", "./local-lb-pems"))
# Deterministic PEM basename so Generate always writes the same filenames
LB_PEM_BASENAME = os.environ.get("LB_PEM_BASENAME", "lb_current").strip()

# ===== Topology selector & defaults =====
# LB_TOPOLOGY governs UI behavior in Cell 3 ("single" or "multi"). When multi, Cell 3 shows LB_COUNT (default 5).
LB_TOPOLOGY = os.environ.get("LB_TOPOLOGY", "single").strip().lower()  # "single" | "multi"
LB_COUNT    = int(os.environ.get("LB_COUNT", "1"))  # Cell 3 will set to 1 for single, or user value for multi

# Node counts & shapes (adjustable in Cell 3)
BACKEND_COUNT = int(os.environ.get("BACKEND_COUNT", "8"))
GENERATOR_COUNT = int(os.environ.get("GENERATOR_COUNT", "8"))

BACKEND_SHAPE  = os.environ.get("BACKEND_SHAPE",  "VM.Standard.E5.Flex")
BACKEND_OCPUS  = float(os.environ.get("BACKEND_OCPUS",  "16"))
BACKEND_MEMORY_GB = float(os.environ.get("BACKEND_MEMORY_GB", "64"))

GENERATOR_SHAPE  = os.environ.get("GENERATOR_SHAPE",  "VM.Standard.E5.Flex")
GENERATOR_OCPUS  = float(os.environ.get("GENERATOR_OCPUS",  "16"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "64"))

# Per‑LB bandwidth caps (Mbps) — used by tfvars in Cell 6
LB_MIN_MBPS = int(os.environ.get("LB_MIN_MBPS", "8000"))
LB_MAX_MBPS = int(os.environ.get("LB_MAX_MBPS", "8000"))

# ===== IPv6 Frontend (LB VIP) and Control‑plane (optional) =====
ENABLE_IPV6_FRONTEND = os.environ.get("ENABLE_IPV6_FRONTEND", "true").strip().lower() in ("1","true","yes","y")
USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")
MASTER_IPV6_OVERRIDE       = os.environ.get("MASTER_IPV6_OVERRIDE",       "").strip()
MASTER_PRIVATE_IP_OVERRIDE = os.environ.get("MASTER_PRIVATE_IP_OVERRIDE", "").strip()

# ===== Endpoints and Locust behavior =====
HEALTH_ENDPOINT_PATH     = os.environ.get("HEALTH_ENDPOINT_PATH",     "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")
LOCUST_WAIT_TIME_SEC   = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
LOCUST_CONNECT_TIMEOUT = int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))  # ms
LOCUST_READ_TIMEOUT    = int(os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000"))   # ms
LOCUST_VERIFY_TLS      = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"

# Remote workspace (must NOT be named "locust")
LOCUST_WORKDIR = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")

# ===== Worker policy =====
EXPECT_WORKERS_STRICT = (os.environ.get("EXPECT_WORKERS_STRICT", "false").lower() == "true")
EXPECTED_WORKERS_OVERRIDE = (None if os.environ.get("EXPECTED_WORKERS_OVERRIDE", "").strip() == "" else int(os.environ.get("EXPECTED_WORKERS_OVERRIDE")))
GRACE_SEC = int(os.environ.get("GRACE_SEC", "60"))

# ===== UI =====
UI_ENABLE      = True
UI_EXPOSE_MODE = "tunnel"  # "tunnel" or "nsg"
UI_WEB_HOST    = "0.0.0.0"  # UI binds on IPv4; SSH tunnel recommended
UI_WEB_PORT    = int(os.environ.get("UI_WEB_PORT", "8089"))
UI_ALLOWED_CIDR = os.environ.get("UI_ALLOWED_CIDR", "0.0.0.0/0")

# ===== Test Mode selector =====
TEST_MODE = os.environ.get("TEST_MODE", "cps").strip().lower()  # "cps" or "throughput"

# ===== CPS tiers & durations (overridable in Cell 3) =====
CPS_TIERS   = [10000, 25000, 35000, 50000, 100000]
CPS_WARMUPS = {10000: 120, 25000: 150, 35000: 160, 50000: 180, 100000: 210}
CPS_HOLDS   = {10000: 600, 25000: 600, 35000: 600, 50000: 600, 100000: 600}

# ===== Throughput configuration defaults =====
TPUT_TARGETS_GBPS_TEXT = os.environ.get("TPUT_TARGETS_GBPS_TEXT", "1,5,10")
TPUT_WARMUP_SEC = int(os.environ.get("TPUT_WARMUP_SEC", "120"))
TPUT_HOLD_SEC   = int(os.environ.get("TPUT_HOLD_SEC",   "600"))

# Payload knobs (finalized in Cell 3)
TPUT_PAYLOAD_SIZE_TEXT  = os.environ.get("TPUT_PAYLOAD_SIZE_TEXT",  "100k")
TPUT_PAYLOAD_SIZES_TEXT = os.environ.get("TPUT_PAYLOAD_SIZES_TEXT", "4k,5k,10k,50k,100k,256k,1m,5m")

# Derived (initialized; resolved in Cell 3)
TPUT_PAYLOAD_SIZE_BYTES = 100_000
TPUT_PAYLOAD_SIZE_LABEL = "100k"
TPUT_PAYLOAD_BYTES_PER_REQ = TPUT_PAYLOAD_SIZE_BYTES

# ===== Generator CPU → worker policy =====
WORKERS_PER_HOST     = os.environ.get("WORKERS_PER_HOST", "auto")  # "auto" or integer/fixed
CPU_RESERVE          = int(os.environ.get("CPU_RESERVE", "1"))
MIN_WORKERS_PER_HOST = int(os.environ.get("MIN_WORKERS_PER_HOST", "1"))
MAX_WORKERS_PER_HOST = int(os.environ.get("MAX_WORKERS_PER_HOST", "32"))

# ===== Misc/overrides =====
MASTER_PRIVATE_IP_OVERRIDE = MASTER_PRIVATE_IP_OVERRIDE  # keep name for orchestrators
SSH_ALLOWED_CIDR = os.environ.get("SSH_ALLOWED_CIDR", "0.0.0.0/0")

# ===== SSH IPv6 CIDR (for NSG ingress; used by Cell 6 tfvars) =====
SSH_ALLOWED_V6_CIDR = os.environ.get("SSH_ALLOWED_V6_CIDR", "::/0").strip()

# ===== Output directory & UTC timestamp =====
OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
TS_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# ===== Placeholders populated in Cell 3 (UI Apply) =====
SELECTED_REGION = None
COMPARTMENT_ID = ""
AD_A = ""
AD_B = ""  # kept for UI parity
IMAGE_ID = ""
SSH_PUBLIC_KEY_CONTENT = ""

# ===== Helpers (do not print secrets) =====
def _ensure_path_exists(path: str, label: str, optional=False):
    if not path:
        if optional:
            return
        raise FileNotFoundError(f"{label} path not set.")
    p = os.path.expanduser(path)
    if not os.path.exists(p):
        if optional:
            return
        raise FileNotFoundError(f"{label} not found: {p}")

# Ensure OCI key is present (fail fast but do not log secret content)
_ensure_path_exists(OCI_PRIVATE_KEY_PATH, "OCI private key", optional=False)

# ===== Lazy certificate generator (NOT executed here; used by Cell 3 when LB_CERT_MODE == "generate") =====
# Keeps notebook self‑contained without adding dependencies unless used.

def generate_self_signed_lb_pems(kind: str, curve_name: str, rsa_bits: int, cn: str, days: int, out_dir: str):
    """
    Generate self-signed PEMs for LB TLS offload.
    Returns (cert_path, key_path, ca_path).
    Uses deterministic filenames via LB_PEM_BASENAME (overwrites on each generate).
    """
    try:
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes, serialization
        from cryptography.hazmat.primitives.asymmetric import ec, rsa
        from cryptography.hazmat.backends import default_backend
        from datetime import datetime as dt2, timedelta
    except ImportError:
        import sys, subprocess
        print("Installing cryptography ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "cryptography"])\
        
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes, serialization
        from cryptography.hazmat.primitives.asymmetric import ec, rsa
        from cryptography.hazmat.backends import default_backend
        from datetime import datetime as dt2, timedelta

    kind = (kind or "ecdsa").strip().lower()
    out_dir = os.path.expanduser(out_dir or "./local-lb-pems")
    os.makedirs(out_dir, exist_ok=True)

    if kind == "ecdsa":
        curve_l = (curve_name or "secp256r1").strip().lower()
        curve_map = {"secp256r1": ec.SECP256R1(), "prime256v1": ec.SECP256R1(), "p-256": ec.SECP256R1()}
        curve = curve_map.get(curve_l, ec.SECP256R1())
        key = ec.generate_private_key(curve, backend=default_backend())
        kind_tag = "ecdsa"
    else:
        key = rsa.generate_private_key(public_exponent=65537, key_size=int(rsa_bits or 2048), backend=default_backend())
        kind_tag = f"rsa{int(rsa_bits or 2048)}"

    subject = issuer = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, cn or "lb.local")])
    builder = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(issuer)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(dt2.utcnow())
        .not_valid_after(dt2.utcnow() + timedelta(days=int(days or 3650)))
        .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
    )
    try:
        builder = builder.add_extension(x509.SubjectAlternativeName([x509.DNSName(cn or "lb.local")]), critical=False)
    except Exception:
        pass

    cert = builder.sign(private_key=key, algorithm=hashes.SHA256(), backend=default_backend())

    key_pem = key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
    cert_pem = cert.public_bytes(serialization.Encoding.PEM)

    base = (LB_PEM_BASENAME if LB_PEM_BASENAME else f"lb_{kind_tag}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}")
    key_path  = os.path.join(out_dir, f"{base}.key.pem")
    cert_path = os.path.join(out_dir, f"{base}.cert.pem")
    ca_path   = os.path.join(out_dir, f"{base}.ca.pem")  # empty for self-signed

    with open(key_path,  "wb") as f: f.write(key_pem)
    with open(cert_path, "wb") as f: f.write(cert_pem)
    with open(ca_path,   "wb") as f: f.write(b"")

    return cert_path, key_path, ca_path

# ===== Minimal completion message (no secrets printed) =====
print(f"Cell 2 complete: Config loaded from {OCI_CONFIG_FILE} [{OCI_PROFILE}] — region={REGION}")
print("Topology:")
print(f"  LB_TOPOLOGY={LB_TOPOLOGY} | LB_COUNT={LB_COUNT} (Multi default in UI will be 5)")
print("IPv6 options:")
print(f"  ENABLE_IPV6_FRONTEND={ENABLE_IPV6_FRONTEND} | USE_SSH_IPV6={USE_SSH_IPV6}")
print(f"  MASTER_IPV6_OVERRIDE={'(set)' if MASTER_IPV6_OVERRIDE else '(empty)'} | MASTER_PRIVATE_IP_OVERRIDE={'(set)' if MASTER_PRIVATE_IP_OVERRIDE else '(empty)'}")
print(f"SSH_ALLOWED_V6_CIDR={SSH_ALLOWED_V6_CIDR}")
print("NEXT: Run Cell 3 to select topology (Single/Multi), region/compartment/shapes, set TLS PEMs (browse or generate), and Apply selections.")


In [ ]:
# Cell 3 — Objective: Interactive dashboard with Topology (Single/Multi), Cert Mode (Browse/Generate),
# IPv6 frontend + optional control-plane overrides (hidden by default), SSH IPv6 CIDR, and Apply.
# Deterministic PEM support + safe generate; applies all selections centrally.

import os
from pathlib import Path
import oci
from IPython.display import display, HTML
import ipywidgets as widgets
from ipyfilechooser import FileChooser

SECTION_BG = "#f8f9fb"
BORDER = "1px solid #e0e0e0"
PAD = "12px"
CONTAINER_MAX_W = "96%"


def section(title_text, body_widgets):
    title = widgets.HTML(f"<b>{title_text}</b>")
    body = body_widgets if isinstance(body_widgets, widgets.Widget) else widgets.VBox(body_widgets)
    return widgets.VBox(
        [title, body],
        layout=widgets.Layout(width="100%", border=BORDER, padding=PAD, margin="8px 0", background_color=SECTION_BG),
    )


def row(*children, gap="12px"):
    return widgets.HBox(list(children), layout=widgets.Layout(gap=gap, width="100%"))


def vspace(h="6px"):
    return widgets.HTML(f"<div style='height:{h}'></div>")


def build_signer(tenancy, user, fp, key_path, passphrase, cfg):
    return oci.signer.Signer(
        tenancy=tenancy,
        user=user,
        fingerprint=fp,
        private_key_file_location=key_path,
        pass_phrase=passphrase if passphrase else None,
        private_key_content=cfg.get("key_content"),
    )


def cfg_for_region(region_name):
    c = dict(_cfg)
    c["region"] = region_name
    return c

# ===== Region selector (shown once at top) =====
_base_signer = build_signer(TENANCY_OCID, USER_OCID, FINGERPRINT, OCI_PRIVATE_KEY_PATH, PRIVATE_KEY_PASSPHRASE, _cfg)
idc_base = oci.identity.IdentityClient(config=_cfg, signer=_base_signer)
subs = sorted(idc_base.list_region_subscriptions(TENANCY_OCID).data, key=lambda r: r.region_name)
region_options = [(r.region_name, r.region_name) for r in subs]
default_region = (
    REGION if REGION in [r.region_name for r in subs]
    else (next((r.region_name for r in subs if r.is_home_region), subs[0].region_name))
)
region_dd = widgets.Dropdown(options=region_options, value=default_region, description="Region:", layout=widgets.Layout(width="100%"))
reload_btn = widgets.Button(description="Reload", icon="refresh")
reset_btn = widgets.Button(description="Reset", icon="history")
err_out, summary_out = widgets.Output(), widgets.Output()
dynamic_box = widgets.VBox([])

# ===== SSH key dropdowns =====

def discover_files(dirs, exts=None, include_hidden=True):
    out = []
    for d in dirs:
        p = Path(os.path.expanduser(d))
        if not p.exists() or not p.is_dir():
            continue
        for f in p.iterdir():
            if not f.is_file():
                continue
            if not include_hidden and f.name.startswith("."):
                continue
            if exts is not None and not any(str(f).endswith(ext) for ext in exts):
                continue
            out.append(str(f))
    out.sort(key=lambda s: Path(s).stat().st_mtime if Path(s).exists() else 0, reverse=True)
    return out

ssh_dir = os.path.expanduser("~/.ssh")
ssh_pub_candidates = [p for p in discover_files([ssh_dir], exts=[".pub"])]
ssh_priv_candidates = [p for p in discover_files([ssh_dir], exts=None) if not p.endswith(".pub")]
ssh_pub_default = next((p for p in ssh_pub_candidates if p == SSH_PUBLIC_KEY_PATH), (ssh_pub_candidates[0] if ssh_pub_candidates else ""))
ssh_priv_default = next((p for p in ssh_priv_candidates if p == SSH_PRIVATE_KEY_PATH), (ssh_priv_candidates[0] if ssh_priv_candidates else ""))

ssh_pub_dd = widgets.Dropdown(options=[(p, p) for p in ssh_pub_candidates] or [("No *.pub keys found in ~/.ssh", "")], value=ssh_pub_default, description="SSH pub:", layout=widgets.Layout(width="100%"))
ssh_priv_dd = widgets.Dropdown(options=[(p, p) for p in ssh_priv_candidates] or [("No private keys found in ~/.ssh", "")], value=ssh_priv_default, description="SSH priv:", layout=widgets.Layout(width="100%"))

# ===== Cert Mode selector =====
cert_mode_dd = widgets.Dropdown(options=[("Browse PEM files", "browse"), ("Generate self-signed PEMs", "generate")], value=LB_CERT_MODE, description="Cert Mode:")

# Browse widgets
home = os.path.expanduser("~")
fc_cert = FileChooser(path=home, title="Select LB certificate (PEM/CRT/CER)", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
fc_key  = FileChooser(path=home, title="Select LB private key (.key or .pem)", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
fc_ca   = FileChooser(path=home, title="Optional: Select CA/chain PEM", show_hidden=True, use_dir_icons=True, show_only_dirs=False, filter_pattern="*")
show_hidden_cb = widgets.Checkbox(value=True, description="Show hidden files (dot folders)")
jump_path_tb = widgets.Text(value=os.path.expanduser("~"), description="Jump to folder:", layout=widgets.Layout(width="100%"))
jump_btn = widgets.Button(description="Go", icon="sign-in")
for w in (fc_cert, fc_key, fc_ca):
    try:
        w.show_hidden = True
    except Exception:
        pass

def _jump_all(path: str):
    p = os.path.expanduser(path or "")
    for fc in (fc_cert, fc_key, fc_ca):
        try:
            fc.reset(path=p)
        except Exception:
            pass

show_hidden_cb.observe(lambda ch: [setattr(x, "show_hidden", bool(ch["new"])) for x in (fc_cert, fc_key, fc_ca)], names="value")
jump_btn.on_click(lambda _: _jump_all(jump_path_tb.value))
jump_row = row(jump_path_tb, jump_btn)

# Generate widgets
kind_dd = widgets.Dropdown(options=[("ECDSA P-256", "ecdsa"), ("RSA-2048", "rsa")], value=LB_CERT_KIND, description="Kind:")
ecdsa_curve_dd = widgets.Dropdown(options=[("secp256r1 (prime256v1)", "secp256r1")], value=LB_ECDSA_CURVE, description="Curve:")
rsa_bits_dd = widgets.Dropdown(options=[("2048", 2048), ("3072", 3072)], value=LB_RSA_BITS, description="RSA bits:")
cn_in = widgets.Text(value=LB_CERT_CN, description="CN (SAN):")
days_in = widgets.BoundedIntText(value=int(LB_CERT_DAYS), min=1, max=36500, step=1, description="Days valid:")
outdir_in = widgets.Text(value=LB_PEM_OUTPUT_DIR, description="Output dir:", layout=widgets.Layout(width="100%"))
pem_basename_in = widgets.Text(value=os.environ.get("LB_PEM_BASENAME", "lb_current"), description="PEM basename:", layout=widgets.Layout(width="100%"))
overwrite_cb = widgets.Checkbox(value=False, description="Overwrite existing PEMs")

gen_note = widgets.HTML("<small>Generate writes PEMs locally. Deterministic basename overwrites the same files; uncheck Overwrite to reuse existing.</small>")

# ===== Topology & LB settings =====
lb_topology_dd = widgets.Dropdown(options=[("Single", "single"), ("Multi", "multi")], value=(LB_TOPOLOGY if LB_TOPOLOGY in ("single","multi") else "single"), description="LB Topology:")
# IntText: no hard max; we validate >=1 on Apply; default 5 for Multi, 1 for Single
lb_count_in = widgets.IntText(value=(5 if (LB_TOPOLOGY == "multi") else 1), description="LB count:")
lb_limits_note = widgets.HTML("<small>OCI resource limits apply; set LB count according to your tenancy quotas.</small>")

lb_min_in = widgets.BoundedIntText(value=int(LB_MIN_MBPS), min=10, max=32000, step=10, description="LB min (Mbps):")
lb_max_in = widgets.BoundedIntText(value=int(LB_MAX_MBPS), min=10, max=32000, step=10, description="LB max (Mbps):")
lb_count_label_single = widgets.HTML("<i>LB count is fixed to 1 in Single topology.</i>")

# ===== IPv6 Frontend & Control‑plane options =====
ipv6_frontend_cb = widgets.Checkbox(value=ENABLE_IPV6_FRONTEND, description="Enable IPv6 frontend (LB VIP)")
ssh_ipv6_cb = widgets.Checkbox(value=USE_SSH_IPV6, description="Use IPv6 for SSH to generators (Cell 9)")
show_overrides_cb = widgets.Checkbox(value=False, description="Show control‑plane override fields (optional)")
master_ipv6_lbl = widgets.HTML("MASTER_IPV6_OVERRIDE:")
master_ipv6_tb = widgets.Text(value=MASTER_IPV6_OVERRIDE, placeholder="Leave empty unless needed", layout=widgets.Layout(width="100%"))
master_ipv4_lbl = widgets.HTML("MASTER_PRIVATE_IP_OVERRIDE:")
master_ipv4_tb = widgets.Text(value=MASTER_PRIVATE_IP_OVERRIDE, placeholder="Leave empty unless needed", layout=widgets.Layout(width="100%"))
ssh_v6_cidr_in = widgets.Text(value=os.environ.get("SSH_ALLOWED_V6_CIDR", "::/0"), description="SSH IPv6 CIDR:", layout=widgets.Layout(width="100%"))

overrides_box = widgets.VBox([row(master_ipv6_lbl, master_ipv6_tb), vspace(), row(master_ipv4_lbl, master_ipv4_tb)])
overrides_box.layout.display = "none"

def _toggle_overrides(_):
    overrides_box.layout.display = "block" if show_overrides_cb.value else "none"
show_overrides_cb.observe(_toggle_overrides, names="value")

ipv6_note = widgets.HTML(
    """
    <i>IPv6 frontend sets the VIP path (Cells 8/10). SSH over IPv6 requires an NSG ingress rule for TCP/22 on nsg‑generators
    from your IPv6 admin CIDR. IPv6 VIP support may vary by region/tenancy; if unavailable, IPv4 VIP will be used.</i>
    """
)

# ===== Locust basics and endpoints =====
health_ep_in = widgets.Text(value=HEALTH_ENDPOINT_PATH, description="Health Path:", layout=widgets.Layout(width="100%"))
throughput_ep_in = widgets.Text(value=THROUGHPUT_ENDPOINT_PATH, description="Throughput Base Path (payload):", layout=widgets.Layout(width="100%"))

test_mode_dd = widgets.Dropdown(options=[("CPS (connections/sec)", "cps"), ("Throughput (Gbps via payload)", "throughput")], value=TEST_MODE, description="Test Mode:")
wait_time_in = widgets.FloatText(value=LOCUST_WAIT_TIME_SEC, description="Wait(s)/user:", step=0.1)
conn_timeout_in = widgets.BoundedIntText(value=LOCUST_CONNECT_TIMEOUT, min=1000, max=60000, step=500, description="Connect ms:")
read_timeout_in = widgets.BoundedIntText(value=LOCUST_READ_TIMEOUT, min=1000, max=120000, step=500, description="Read ms:")
verify_tls_in = widgets.Checkbox(value=LOCUST_VERIFY_TLS, description="Verify TLS (False is recommended for VIP IP)")

# ===== UI params =====
ui_enable_in = widgets.Checkbox(value=UI_ENABLE, description="Enable Locust UI")
ui_mode_in = widgets.Dropdown(options=["tunnel", "nsg"], value=UI_EXPOSE_MODE, description="UI expose mode:")
ui_port_in = widgets.BoundedIntText(value=UI_WEB_PORT, min=1024, max=65535, step=1, description="UI port:")
ui_cidr_in = widgets.Text(value=UI_ALLOWED_CIDR, description="UI allowed CIDR:")

# ===== CPU policy =====
workers_per_host_mode_in = widgets.Dropdown(options=[("Auto (use OCPUs)", "auto"), ("Fixed count", "fixed")], value=("auto" if str(WORKERS_PER_HOST).lower() == "auto" else "fixed"), description="Workers/host mode:")
fixed_workers_in = widgets.BoundedIntText(value=(int(MIN_WORKERS_PER_HOST) if str(WORKERS_PER_HOST).lower() == "fixed" else 16), min=1, max=256, step=1, description="Fixed workers/host:")
cpu_reserve_in = widgets.BoundedIntText(value=int(CPU_RESERVE), min=0, max=8, step=1, description="CPU reserve:")
min_workers_in = widgets.BoundedIntText(value=int(MIN_WORKERS_PER_HOST), min=1, max=256, step=1, description="Min workers/host:")
max_workers_in = widgets.BoundedIntText(value=int(MAX_WORKERS_PER_HOST), min=1, max=512, step=1, description="Max workers/host:")

# ===== Counts and shapes =====
backend_count_in = widgets.BoundedIntText(value=int(BACKEND_COUNT), min=1, max=64, step=1, description="Backends:")
generator_count_in = widgets.BoundedIntText(value=int(GENERATOR_COUNT), min=1, max=128, step=1, description="Generators:")

# ===== CPS tiers/durations =====

def _int_box(v, desc):
    return widgets.BoundedIntText(value=int(v), min=1, max=36000, step=1, description=desc)

warm_10k_in, hold_10k_in   = _int_box(CPS_WARMUPS[10000], "10k warmup(s):"), _int_box(CPS_HOLDS[10000], "10k hold(s):")
warm_25k_in, hold_25k_in   = _int_box(CPS_WARMUPS[25000], "25k warmup(s):"), _int_box(CPS_HOLDS[25000], "25k hold(s):")
warm_35k_in, hold_35k_in   = _int_box(CPS_WARMUPS[35000], "35k warmup(s):"), _int_box(CPS_HOLDS[35000], "35k hold(s):")
warm_50k_in, hold_50k_in   = _int_box(CPS_WARMUPS[50000], "50k warmup(s):"), _int_box(CPS_HOLDS[50000], "50k hold(s):")
warm_100k_in, hold_100k_in = _int_box(CPS_WARMUPS[100000], "100k warmup(s):"), _int_box(CPS_HOLDS[100000], "100k hold(s):")

# ===== Throughput widgets =====
tput_targets_in = widgets.Text(value=TPUT_TARGETS_GBPS_TEXT, description="TPUT targets (Gbps):", layout=widgets.Layout(width="100%"))
tput_warm_in = widgets.BoundedIntText(value=int(TPUT_WARMUP_SEC), min=1, max=36000, step=1, description="TPUT warmup(s):")
tput_hold_in = widgets.BoundedIntText(value=int(TPUT_HOLD_SEC), min=1, max=36000, step=1, description="TPUT hold(s):")
tput_payload_size_in = widgets.Text(value=TPUT_PAYLOAD_SIZE_TEXT, description="TPUT payload for run (e.g., 4k, 10k, 100k, 1m):")
tput_payload_bake_in = widgets.Text(value=TPUT_PAYLOAD_SIZES_TEXT, description="Payload sizes to bake (comma-separated):", layout=widgets.Layout(width="100%"))
payload_help = widgets.HTML("<i>Suffixes: k ≈1000 bytes, m ≈1,000,000 bytes. Examples: 4k, 5k, 10k, 100k, 1m, 5m.</i>")

# ===== Buttons =====
apply_btn = widgets.Button(description="Apply Selections", button_style="primary", icon="check", layout=widgets.Layout(width="240px", height="36px", align_self="center"))

_state = {
    "comp_dd": None,
    "ad_a_dd": None,
    "shape_filter": None,
    "backend_shape_dd": None,
    "generator_shape_dd": None,
    "image_dd": None,
    "backend_ocpus_in": None,
    "backend_mem_in": None,
    "generator_ocpus_in": None,
    "generator_mem_in": None,
}

# ===== OCI helpers =====

def identity_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(TENANCY_OCID, USER_OCID, FINGERPRINT, OCI_PRIVATE_KEY_PATH, PRIVATE_KEY_PASSPHRASE, cfg_r)
    return oci.identity.IdentityClient(config=cfg_r, signer=signer)


def compute_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(TENANCY_OCID, USER_OCID, FINGERPRINT, OCI_PRIVATE_KEY_PATH, PRIVATE_KEY_PASSPHRASE, cfg_r)
    return oci.core.ComputeClient(config=cfg_r, signer=signer)


def list_compartments(idc):
    comps = oci.pagination.list_call_get_all_results(idc.list_compartments, TENANCY_OCID, compartment_id_in_subtree=True, access_level="ACCESSIBLE").data
    comps = [c for c in comps if c.lifecycle_state == "ACTIVE"]
    tenancy = idc.get_tenancy(TENANCY_OCID).data
    tenancy_name = getattr(tenancy, "name", "root-tenancy")
    return [(f"{tenancy_name} (root)", TENANCY_OCID)] + sorted([(c.name + f" ({c.description or 'no-desc'})", c.id) for c in comps], key=lambda t: t[0].lower())


def list_ads(idc):
    ads = oci.pagination.list_call_get_all_results(idc.list_availability_domains, TENANCY_OCID).data
    return sorted([ad.name for ad in ads]) or ["AD-1"]


def list_shapes(cc):
    shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
    return sorted({s.shape for s in shapes})


def list_images(cc, comp_id: str, b_shape: str, g_shape: str):
    def imgs_for(shape):
        return oci.pagination.list_call_get_all_results(cc.list_images, comp_id, operating_system="Oracle Linux", sort_by="TIMECREATED", sort_order="DESC", shape=shape).data
    if b_shape and g_shape:
        b, g = imgs_for(b_shape), imgs_for(g_shape)
        g_ids = {im.id for im in g}
        return [im for im in b if im.id in g_ids]
    shape = b_shape or g_shape
    return (imgs_for(shape) if shape else oci.pagination.list_call_get_all_results(cc.list_images, comp_id, operating_system="Oracle Linux", sort_by="TIMECREATED", sort_order="DESC").data)


def show_flex_inputs(back_shape: str, gen_shape: str):
    if _state["backend_ocpus_in"] is None:
        _state["backend_ocpus_in"] = widgets.BoundedIntText(value=int(BACKEND_OCPUS), min=1, max=128, step=1, description="Backend OCPUs:")
        _state["backend_mem_in"] = widgets.BoundedIntText(value=int(BACKEND_MEMORY_GB), min=1, max=2048, step=1, description="Backend Memory(GB):")
        _state["generator_ocpus_in"] = widgets.BoundedIntText(value=int(GENERATOR_OCPUS), min=1, max=256, step=1, description="Generator OCPUs:")
        _state["generator_mem_in"] = widgets.BoundedIntText(value=int(GENERATOR_MEMORY_GB), min=1, max=4096, step=1, description="Generator Memory(GB):")
    _state["backend_ocpus_in"].layout.display   = ("block" if (back_shape or "").endswith(".Flex") else "none")
    _state["backend_mem_in"].layout.display     = ("block" if (back_shape or "").endswith(".Flex") else "none")
    _state["generator_ocpus_in"].layout.display = ("block" if (gen_shape or "").endswith(".Flex") else "none")
    _state["generator_mem_in"].layout.display   = ("block" if (gen_shape or "").endswith(".Flex") else "none")

# ===== Payload parsing helpers =====

def _parse_size_text_to_bytes(s: str) -> tuple[int, str]:
    s = (s or "").strip().lower()
    if not s: raise ValueError("Empty payload size")
    if s.endswith("mb"): s = s[:-2] + "m"
    if s.endswith("kb"): s = s[:-2] + "k"
    if s.endswith("m"):
        n = float(s[:-1]);
        if n <= 0: raise ValueError("MB value must be > 0")
        return int(n * 1_000_000), f"{int(n) if n.is_integer() else n}m"
    if s.endswith("k"):
        n = float(s[:-1]);
        if n <= 0: raise ValueError("KB value must be > 0")
        return int(n * 1_000), f"{int(n) if n.is_integer() else n}k"
    n = float(s)
    if n <= 0: raise ValueError("Byte value must be > 0")
    return int(n), str(int(n))


def _normalize_bake_list(txt: str) -> list[str]:
    out = []
    for tok in (txt or "").split(","):
        tok = tok.strip()
        if not tok: continue
        _, label = _parse_size_text_to_bytes(tok)
        out.append(label)
    seen, norm = set(), []
    for l in out:
        if l in seen: continue
        seen.add(l); norm.append(l)
    return norm

# ===== Dynamic UI rebuild =====

def rebuild_dynamic_area(_=None):
    err_out.clear_output()
    with err_out:
        print(f"Refreshing for region: {region_dd.value} ...")
    try:
        idc = identity_client_for_current()
        cc = compute_client_for_current()

        comp_opts = list_compartments(idc)
        comp_dd = widgets.Dropdown(options=comp_opts, value=comp_opts[0][1], description="Compartment:", layout=widgets.Layout(width="100%"))
        comp_filter = widgets.Text(value="", description="Comp filter:", placeholder="substring (optional)", layout=widgets.Layout(width="100%"))

        def on_comp_filter_change(_ch):
            text = comp_filter.value.strip().lower()
            filtered = (comp_opts if not text else [o for o in comp_opts if text in o[0].lower()])
            comp_dd.options = filtered or comp_opts
            comp_dd.value = (filtered or comp_opts)[0][1]
            refresh_images()
        comp_filter.observe(on_comp_filter_change, names="value")

        ad_names = list_ads(idc)
        ad_a_dd = widgets.Dropdown(options=[(n, n) for n in ad_names], value=ad_names[0], description="LB AD:", layout=widgets.Layout(width="100%"))
        _state["comp_dd"], _state["ad_a_dd"] = comp_dd, ad_a_dd

        all_shapes = list_shapes(cc)
        shape_filter = widgets.Text(value="", description="Shape filter:", placeholder="e.g. E5.Flex", layout=widgets.Layout(width="100%"))

        def filtered_shapes():
            if not shape_filter.value.strip(): return all_shapes
            s = shape_filter.value.strip().lower()
            return [n for n in all_shapes if s in n.lower()]

        backend_shape_dd = widgets.Dropdown(options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")], value=(filtered_shapes()[0] if filtered_shapes() else ""), description="Backend Shape:", layout=widgets.Layout(width="100%"))
        generator_shape_dd = widgets.Dropdown(options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")], value=(filtered_shapes()[0] if filtered_shapes() else ""), description="Generator Shape:", layout=widgets.Layout(width="100%"))
        _state.update({"shape_filter": shape_filter, "backend_shape_dd": backend_shape_dd, "generator_shape_dd": generator_shape_dd})

        def on_shape_filter(_ch):
            opts = filtered_shapes()
            backend_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            backend_shape_dd.value = opts[0] if opts else ""
            generator_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            generator_shape_dd.value = opts[0] if opts else ""
            show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value)
            refresh_images()
        shape_filter.observe(on_shape_filter, names="value")
        backend_shape_dd.observe(lambda _ch: (show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value), refresh_images()), names="value")
        generator_shape_dd.observe(lambda _ch: (show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value), refresh_images()), names="value")
        show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value)

        image_dd = widgets.Dropdown(options=[("Select compartment first", "")], value="", description="Image:", layout=widgets.Layout(width="100%"))
        _state["image_dd"] = image_dd

        def refresh_images():
            try:
                comp_id = comp_dd.value
                b, g = backend_shape_dd.value or "", generator_shape_dd.value or ""
                imgs = list_images(cc, comp_id, b, g)
                items = []
                for im in imgs[:100]:
                    try: created = im.time_created.strftime("%Y-%m-%d")
                    except: created = ""
                    items.append((f"{im.display_name} — {im.operating_system} {im.operating_system_version} — {created}", im.id))
                image_dd.options = items or [("No compatible Oracle Linux images for current shape(s))", "")]
                image_dd.value = image_dd.options[0][1] if image_dd.options else ""
            except Exception:
                image_dd.options = [("Discovery error; try Reload/Reset", "")]
                image_dd.value = ""
        refresh_images()

        # Compose UI sections
        topology_box = section("Topology", [row(lb_topology_dd), vspace(), row(lb_count_in), lb_limits_note])

        location_box = section("Location", [row(comp_filter), vspace(), row(comp_dd), vspace(), row(ad_a_dd)])
        shapes_box = section("Shapes and Flex configuration", [
            row(shape_filter), vspace(), row(backend_shape_dd), vspace(),
            row(backend_count_in, _state["backend_ocpus_in"] or widgets.Label(""), _state["backend_mem_in"] or widgets.Label("")),
            vspace(), row(generator_shape_dd), vspace(),
            row(generator_count_in, _state["generator_ocpus_in"] or widgets.Label(""), _state["generator_mem_in"] or widgets.Label("")),
            vspace(), row(image_dd),
        ])

        browse_box = section("LB TLS — Browse PEMs", [
            widgets.HTML("<i>Provide existing certificate and private key (optional CA chain).</i>"),
            vspace("6px"), show_hidden_cb, row(jump_row), vspace("6px"), fc_cert, vspace("6px"), fc_key, vspace("6px"), fc_ca,
        ])
        generate_box = section("LB TLS — Generate self‑signed PEMs", [
            row(kind_dd, ecdsa_curve_dd, rsa_bits_dd), vspace(), row(cn_in, days_in), vspace(), row(outdir_in), vspace(), row(pem_basename_in), row(overwrite_cb), gen_note,
        ])

        def _toggle_cert_ui(mode: str):
            m = (mode or "browse").lower()
            browse_box.layout.display = "block" if m == "browse" else "none"
            generate_box.layout.display = "block" if m == "generate" else "none"
            ecdsa_curve_dd.layout.display = ("block" if (kind_dd.value == "ecdsa" and m == "generate") else "none")
            rsa_bits_dd.layout.display   = ("block" if (kind_dd.value == "rsa"   and m == "generate") else "none")
        cert_mode_dd.observe(lambda ch: _toggle_cert_ui(ch["new"]), names="value")
        kind_dd.observe(lambda ch: _toggle_cert_ui(cert_mode_dd.value), names="value")
        _toggle_cert_ui(cert_mode_dd.value)

        lb_settings_box = section("Load Balancer Settings", [
            widgets.HTML("<i>Flexible LB bandwidth per LB. In Multi, requests fan out across all VIPs via LOCUST_TARGETS.</i>"),
            vspace("6px"), row(lb_min_in, lb_max_in), vspace("6px"), lb_count_label_single,
        ])

        cert_mode_box = section("Certificate Mode", [cert_mode_dd])

        cps_box = section("CPS settings (used when Test Mode=CPS)", [
            row(warm_10k_in, hold_10k_in), vspace(), row(warm_25k_in, hold_25k_in), vspace(),
            row(warm_35k_in, hold_35k_in), vspace(), row(warm_50k_in, hold_50k_in), vspace(), row(warm_100k_in, hold_100k_in),
        ])
        tput_box = section("Throughput settings (used when Test Mode=Throughput)", [
            row(tput_targets_in), vspace(), row(tput_warm_in, tput_hold_in), vspace(), row(tput_payload_size_in), vspace(), row(tput_payload_bake_in), payload_help,
        ])
        cpu_box = section("Generator CPU policy (workers per host)", [
            row(workers_per_host_mode_in, fixed_workers_in), vspace(), row(cpu_reserve_in, min_workers_in, max_workers_in),
            widgets.HTML("<i>Auto mode: workers/host = clamp(nproc − CPU_RESERVE, MIN, MAX).</i>"),
        ])
        cps_locust_box = section("Locust & UI settings", [
            row(test_mode_dd), vspace(), row(wait_time_in, conn_timeout_in, read_timeout_in, verify_tls_in), vspace(), row(health_ep_in), vspace(), row(throughput_ep_in), vspace(), row(ui_enable_in, ui_mode_in, ui_port_in, ui_cidr_in),
        ])
        ipv6_box = section("IPv6 Frontend & Control‑plane", [
            row(ipv6_frontend_cb, ssh_ipv6_cb), show_overrides_cb, overrides_box, row(ssh_v6_cidr_in), ipv6_note,
        ])

        dynamic_box.children = [
            topology_box, location_box, shapes_box, lb_settings_box, cert_mode_box, browse_box, generate_box, ipv6_box, cps_box, tput_box, cpu_box, cps_locust_box,
        ]

        # Topology-driven visibility
        def _apply_topology_visibility():
            if lb_topology_dd.value == "single":
                lb_count_in.layout.display = "none"
                lb_limits_note.layout.display = "none"
                lb_count_label_single.layout.display = "block"
            else:
                lb_count_in.layout.display = "block"
                lb_limits_note.layout.display = "block"
                lb_count_label_single.layout.display = "none"
        _apply_topology_visibility()
        lb_topology_dd.observe(lambda ch: _apply_topology_visibility(), names="value")

        err_out.clear_output()
    except Exception as e:
        dynamic_box.children = []
        with err_out:
            print("[error] UI rebuild failed:", repr(e))
            print("Use Reset, then try again. If it persists, restart the kernel and run Cells 1–3.")

# ===== Apply handler =====

def on_reload_clicked(_):
    rebuild_dynamic_area()

def on_reset_clicked(_):
    region_dd.value = default_region
    rebuild_dynamic_area()

def on_apply_clicked(_):
    try:
        global SELECTED_REGION, REGION, COMPARTMENT_ID, AD_A, AD_B, IMAGE_ID
        global SSH_PUBLIC_KEY_PATH, SSH_PRIVATE_KEY_PATH, SSH_PUBLIC_KEY_CONTENT
        global LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH, LB_CERT_MODE
        global BACKEND_COUNT, BACKEND_SHAPE, BACKEND_OCPUS, BACKEND_MEMORY_GB
        global GENERATOR_COUNT, GENERATOR_SHAPE, GENERATOR_OCPUS, GENERATOR_MEMORY_GB
        global LB_MIN_MBPS, LB_MAX_MBPS, LB_TOPOLOGY, LB_COUNT
        global HEALTH_ENDPOINT_PATH, THROUGHPUT_ENDPOINT_PATH
        global LOCUST_WAIT_TIME_SEC, LOCUST_CONNECT_TIMEOUT, LOCUST_READ_TIMEOUT, LOCUST_VERIFY_TLS
        global UI_ENABLE, UI_EXPOSE_MODE, UI_WEB_PORT, UI_ALLOWED_CIDR
        global TEST_MODE, CPS_WARMUPS, CPS_HOLDS
        global TPUT_TARGETS_GBPS_TEXT, TPUT_WARMUP_SEC, TPUT_HOLD_SEC
        global TPUT_PAYLOAD_SIZE_TEXT, TPUT_PAYLOAD_SIZE_BYTES, TPUT_PAYLOAD_SIZE_LABEL
        global TPUT_PAYLOAD_SIZES_TEXT, TPUT_PAYLOAD_BYTES_PER_REQ
        global LB_CERT_KIND, LB_ECDSA_CURVE, LB_RSA_BITS, LB_CERT_CN, LB_CERT_DAYS, LB_PEM_OUTPUT_DIR
        global LB_PEM_BASENAME
        global ENABLE_IPV6_FRONTEND, USE_SSH_IPV6, MASTER_IPV6_OVERRIDE, MASTER_PRIVATE_IP_OVERRIDE
        global WORKERS_PER_HOST, CPU_RESERVE, MIN_WORKERS_PER_HOST, MAX_WORKERS_PER_HOST

        SELECTED_REGION = region_dd.value
        REGION = SELECTED_REGION
        cc = compute_client_for_current()  # validates region context

        backend_shape_dd = _state["backend_shape_dd"]
        generator_shape_dd = _state["generator_shape_dd"]
        image_dd = _state["image_dd"]
        comp_dd = _state["comp_dd"]
        ad_a_dd = _state["ad_a_dd"]

        COMPARTMENT_ID = comp_dd.value
        AD_A = ad_a_dd.value
        AD_B = ad_a_dd.value

        BACKEND_COUNT = int(backend_count_in.value)
        GENERATOR_COUNT = int(generator_count_in.value)

        BACKEND_SHAPE = backend_shape_dd.value or ""
        GENERATOR_SHAPE = generator_shape_dd.value or ""
        if BACKEND_SHAPE.endswith(".Flex"):
            BACKEND_OCPUS = float(_state["backend_ocpus_in"].value)
            BACKEND_MEMORY_GB = float(_state["backend_mem_in"].value)
        if GENERATOR_SHAPE.endswith(".Flex"):
            GENERATOR_OCPUS = float(_state["generator_ocpus_in"].value)
            GENERATOR_MEMORY_GB = float(_state["generator_mem_in"].value)

        IMAGE_ID = image_dd.value or ""

        # SSH keys
        SSH_PUBLIC_KEY_PATH = os.path.expanduser(ssh_pub_dd.value or SSH_PUBLIC_KEY_PATH)
        SSH_PRIVATE_KEY_PATH = os.path.expanduser(ssh_priv_dd.value or SSH_PRIVATE_KEY_PATH)
        with open(SSH_PUBLIC_KEY_PATH, "r") as f:
            SSH_PUBLIC_KEY_CONTENT = f.read().strip()

        # Topology & LB Settings
        LB_TOPOLOGY = lb_topology_dd.value
        if LB_TOPOLOGY == "single":
            LB_COUNT = 1
        else:
            n = int(lb_count_in.value or 1)
            if n < 1:
                raise ValueError("LB count must be >= 1")
            LB_COUNT = n
        LB_MIN_MBPS = int(lb_min_in.value)
        LB_MAX_MBPS = int(lb_max_in.value)

        # Cert Mode handling
        LB_CERT_MODE = cert_mode_dd.value
        if LB_CERT_MODE == "browse":
            LB_CERT_PEM_PATH = os.path.expanduser(fc_cert.selected or "")
            LB_KEY_PEM_PATH  = os.path.expanduser(fc_key.selected or "")
            LB_CA_PEM_PATH   = os.path.expanduser(fc_ca.selected or "")
            if not LB_CERT_PEM_PATH or not os.path.exists(LB_CERT_PEM_PATH):
                raise ValueError("LB cert PEM missing. Use the 'Select LB certificate' picker.")
            if not LB_KEY_PEM_PATH or not os.path.exists(LB_KEY_PEM_PATH):
                raise ValueError("LB key PEM missing. Use the 'Select LB private key' picker.")
        else:
            LB_CERT_KIND = kind_dd.value
            LB_ECDSA_CURVE = ecdsa_curve_dd.value
            LB_RSA_BITS = int(rsa_bits_dd.value)
            LB_CERT_CN = (cn_in.value or "lb.local").strip()
            LB_CERT_DAYS = int(days_in.value)
            LB_PEM_OUTPUT_DIR = os.path.expanduser(outdir_in.value or "./local-lb-pems")
            LB_PEM_BASENAME = (pem_basename_in.value or "lb_current").strip()
            os.environ["LB_PEM_BASENAME"] = LB_PEM_BASENAME
            base_dir = LB_PEM_OUTPUT_DIR
            cert_path = os.path.join(base_dir, f"{LB_PEM_BASENAME}.cert.pem")
            key_path  = os.path.join(base_dir, f"{LB_PEM_BASENAME}.key.pem")
            ca_path   = os.path.join(base_dir, f"{LB_PEM_BASENAME}.ca.pem")
            exists_all = all(os.path.exists(p) for p in (cert_path, key_path, ca_path))
            if exists_all and not overwrite_cb.value:
                LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH = (cert_path, key_path, ca_path)
            else:
                cert_path, key_path, ca_path = generate_self_signed_lb_pems(LB_CERT_KIND, LB_ECDSA_CURVE, LB_RSA_BITS, LB_CERT_CN, LB_CERT_DAYS, LB_PEM_OUTPUT_DIR)
                LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH = (cert_path, key_path, ca_path)

        # Persist PEM paths for Cell 6
        os.environ["LB_CERT_PEM_PATH"] = LB_CERT_PEM_PATH
        os.environ["LB_KEY_PEM_PATH"]  = LB_KEY_PEM_PATH
        os.environ["LB_CA_PEM_PATH"]   = LB_CA_PEM_PATH

        # Endpoints and timeouts
        HEALTH_ENDPOINT_PATH     = (health_ep_in.value or "/healthz").strip()
        THROUGHPUT_ENDPOINT_PATH = (throughput_ep_in.value or "/payload_100k").strip()
        LOCUST_WAIT_TIME_SEC     = float(wait_time_in.value)
        LOCUST_CONNECT_TIMEOUT   = int(conn_timeout_in.value)
        LOCUST_READ_TIMEOUT      = int(read_timeout_in.value)
        LOCUST_VERIFY_TLS        = bool(verify_tls_in.value)

        # Payloads
        TPUT_TARGETS_GBPS_TEXT = (tput_targets_in.value or "1,5,10").strip()
        TPUT_WARMUP_SEC = int(tput_warm_in.value)
        TPUT_HOLD_SEC   = int(tput_hold_in.value)
        TPUT_PAYLOAD_SIZE_TEXT  = (tput_payload_size_in.value or "100k").strip()
        TPUT_PAYLOAD_SIZES_TEXT = (tput_payload_bake_in.value or "4k,5k,10k,50k,100k,256k,1m,5m").strip()
        TPUT_PAYLOAD_SIZE_BYTES, TPUT_PAYLOAD_SIZE_LABEL = _parse_size_text_to_bytes(TPUT_PAYLOAD_SIZE_TEXT)
        TPUT_PAYLOAD_BYTES_PER_REQ = int(TPUT_PAYLOAD_SIZE_BYTES)

        # UI toggles
        UI_ENABLE      = bool(ui_enable_in.value)
        UI_EXPOSE_MODE = ui_mode_in.value or "tunnel"
        UI_WEB_PORT    = int(ui_port_in.value)
        UI_ALLOWED_CIDR = (ui_cidr_in.value or "0.0.0.0/0").strip()

        # CPS tiers
        CPS_WARMUPS = {10000: int(warm_10k_in.value), 25000: int(warm_25k_in.value), 35000: int(warm_35k_in.value), 50000: int(warm_50k_in.value), 100000: int(warm_100k_in.value)}
        CPS_HOLDS   = {10000: int(hold_10k_in.value), 25000: int(hold_25k_in.value), 35000: int(hold_35k_in.value), 50000: int(hold_50k_in.value), 100000: int(hold_100k_in.value)}

        # IPv6 toggles & overrides
        ENABLE_IPV6_FRONTEND = bool(ipv6_frontend_cb.value)
        USE_SSH_IPV6         = bool(ssh_ipv6_cb.value)
        MASTER_IPV6_OVERRIDE = (master_ipv6_tb.value.strip() if show_overrides_cb.value else "")
        MASTER_PRIVATE_IP_OVERRIDE = (master_ipv4_tb.value.strip() if show_overrides_cb.value else "")

        # Persist core env for downstream cells (avoid secret values)
        os.environ["OCI_CONFIG_FILE"] = OCI_CONFIG_FILE
        os.environ["OCI_PROFILE"]     = OCI_PROFILE
        os.environ["REGION"]          = REGION
        os.environ["COMPARTMENT_ID"]  = COMPARTMENT_ID
        os.environ["LB_CERT_MODE"]    = LB_CERT_MODE

        # Persist topology and IPv6 options
        os.environ["LB_TOPOLOGY"] = LB_TOPOLOGY
        os.environ["LB_COUNT"]    = str(LB_COUNT)
        os.environ["ENABLE_IPV6_FRONTEND"] = "true" if ENABLE_IPV6_FRONTEND else "false"
        os.environ["USE_SSH_IPV6"]         = "true" if USE_SSH_IPV6 else "false"
        os.environ["MASTER_IPV6_OVERRIDE"] = MASTER_IPV6_OVERRIDE
        os.environ["MASTER_PRIVATE_IP_OVERRIDE"] = MASTER_PRIVATE_IP_OVERRIDE

        # Persist IPv6 SSH CIDR (used by Cell 6 tfvars)
        os.environ["SSH_ALLOWED_V6_CIDR"] = (ssh_v6_cidr_in.value or "::/0").strip()

        # Persist SSH key paths (ensures later cells/tunnel commands use the selected key)
        os.environ["SSH_PRIVATE_KEY_PATH"] = SSH_PRIVATE_KEY_PATH
        os.environ["SSH_PUBLIC_KEY_PATH"]  = SSH_PUBLIC_KEY_PATH

        def _bn(p):
            try:
                return os.path.basename(p) if p else "(unset)"
            except:
                return "(unset)"

        # Summary
        summary_out.clear_output()
        with summary_out:
            topo_line = (
                f"LB (Single): min/max (Mbps): {LB_MIN_MBPS} / {LB_MAX_MBPS}<br>" if LB_TOPOLOGY == "single"
                else f"LBs: count={LB_COUNT} | min/max (Mbps) per‑LB: {LB_MIN_MBPS} / {LB_MAX_MBPS}<br><small>OCI resource limits apply.</small><br>"
            )
            display(HTML(f"""
<div style="border:{BORDER};background:{SECTION_BG};padding:{PAD};">
  <b>Selections applied</b>
  <div style="font-family:ui-monospace; white-space:pre-wrap;">
Region: {REGION}
Compartment: {COMPARTMENT_ID}
AD: {AD_A}

{topo_line}
LB offload TLS: TCP listener with SSL; PPv2 to backends (HTTP:80)
Cert Mode: {LB_CERT_MODE}
PEM basename: {LB_PEM_BASENAME if LB_CERT_MODE=='generate' else '(n/a)'}
LB cert file: {_bn(LB_CERT_PEM_PATH)}
LB key file:  {_bn(LB_KEY_PEM_PATH)}
LB CA file:   {_bn(LB_CA_PEM_PATH)}

Backend Shape: {BACKEND_SHAPE} | OCPUs={BACKEND_OCPUS if BACKEND_SHAPE.endswith('.Flex') else '-'} | MemGB={BACKEND_MEMORY_GB if BACKEND_SHAPE.endswith('.Flex') else '-'} | Count={BACKEND_COUNT}
Generator Shape: {GENERATOR_SHAPE} | OCPUs={GENERATOR_OCPUS if GENERATOR_SHAPE.endswith('.Flex') else '-'} | MemGB={GENERATOR_MEMORY_GB if GENERATOR_SHAPE.endswith('.Flex') else '-'} | Count={GENERATOR_COUNT}
Image: {IMAGE_ID or '(unset)'}

IPv6: frontend={ENABLE_IPV6_FRONTEND} | ssh_v6={USE_SSH_IPV6}
Overrides: master_ipv6={'(set)' if MASTER_IPV6_OVERRIDE else '(empty)'} | master_ipv4={'(set)' if MASTER_PRIVATE_IP_OVERRIDE else '(empty)'}
SSH IPv6 CIDR: {os.environ.get('SSH_ALLOWED_V6_CIDR','::/0')}

Health path: {HEALTH_ENDPOINT_PATH}
Throughput path (selected payload): {THROUGHPUT_ENDPOINT_PATH}
Throughput payload (run): {TPUT_PAYLOAD_SIZE_LABEL} (~{TPUT_PAYLOAD_SIZE_BYTES} bytes)
Payloads to bake: {_normalize_bake_list(TPUT_PAYLOAD_SIZES_TEXT)}

Locust: wait(s)={LOCUST_WAIT_TIME_SEC} | connect(ms)={LOCUST_CONNECT_TIMEOUT} | read(ms)={LOCUST_READ_TIMEOUT} | verify_tls={LOCUST_VERIFY_TLS}
UI: enable={UI_ENABLE} | mode={UI_EXPOSE_MODE} | port={UI_WEB_PORT} | allowed_cidr={UI_ALLOWED_CIDR}

CPS Warmups: {CPS_WARMUPS}
CPS Holds:   {CPS_HOLDS}

SSH private key: {_bn(SSH_PRIVATE_KEY_PATH)}
  </div>
</div>
"""))
        print("Cell 3 complete: Selections applied; PEMs & IPv6 SSH CIDR ready. Topology persisted.")
        print("- PROVISION infra: run Cells 4 → 7. Then Cells 8 → 12.")
    except Exception as e:
        err_out.clear_output()
        with err_out:
            print("[error] Apply failed:", repr(e))

reload_btn.on_click(on_reload_clicked)
reset_btn.on_click(on_reset_clicked)
apply_btn.on_click(on_apply_clicked)
region_dd.observe(rebuild_dynamic_area, names="value")

top_region = section("Region", [row(region_dd, reload_btn, reset_btn)])
rebuild_dynamic_area()
container = widgets.VBox(
    [
        top_region,
        dynamic_box,
        section("Keys (API + SSH)", [row(ssh_pub_dd), vspace(), row(ssh_priv_dd)]),
        section("Apply", [apply_btn]),
        err_out,
        summary_out,
    ],
    layout=widgets.Layout(width="100%", max_width=CONTAINER_MAX_W, margin="0 auto"),
)

display(container)
print("Cell 3 loaded: Choose Topology (Single/Multi), set TLS PEMs (Browse/Generate), IPv6 options (incl. SSH IPv6 CIDR), complete inputs, then Apply.")


In [ ]:
# Cell 4 — Objective: Write cloud-init scripts (Backends HTTP:80 proxy_protocol; Generators Locust; payloads baked)
# - Backend: proven Single variant (NGINX on 80 with PROXY protocol v2; tuned; payload files pre-baked)
# - Generator: multi-capable locustfile (supports LOCUST_TARGETS cycling + per‑VIP naming), still compatible with Single

import os

os.makedirs("cloud-init", exist_ok=True)


def _dd_lines_for_payloads(sizes_csv: str) -> str:
    lines = []

    def _one(tok: str):
        tok = tok.strip().lower()
        if not tok:
            return
        if tok.endswith("mb"):
            tok2 = tok[:-2] + "m"
        elif tok.endswith("kb"):
            tok2 = tok[:-2] + "k"
        else:
            tok2 = tok
        if tok2.endswith("m"):
            n = tok2[:-1]
            lines.append(
                f"dd if=/dev/zero of=/usr/share/nginx/html/payload_{n}m bs=1MB count={n} status=none || true"
            )
        elif tok2.endswith("k"):
            n = tok2[:-1]
            lines.append(
                f"dd if=/dev/zero of=/usr/share/nginx/html/payload_{n}k bs=1KB count={n} status=none || true"
            )
        else:
            lines.append(
                f"head -c {tok2} /dev/zero > /usr/share/nginx/html/payload_{tok2}b || true"
            )

    for t in (TPUT_PAYLOAD_SIZES_TEXT or "").split(","):
        _one(t)
    if not lines:
        lines.append(
            "dd if=/dev/zero of=/usr/share/nginx/html/payload_100k bs=1KB count=100 status=none || true"
        )
    return "\n".join(lines)


_dd_payload_block = _dd_lines_for_payloads(TPUT_PAYLOAD_SIZES_TEXT)

# Backend cloud-init (from the proven Single IPv6→IPv4 variant)
backend_cloud_init_template = """#!/bin/bash
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi

# Kernel tuning for high CPS
cat <<EOF >/etc/sysctl.d/99-net-tuning.conf
net.core.somaxconn=262144
net.core.netdev_max_backlog=500000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
net.core.rmem_max=33554432
net.core.wmem_max=33554432
net.ipv4.tcp_syncookies=1
net.ipv4.tcp_fastopen=3
# Additional safe CPS-focused tweaks
net.ipv4.tcp_abort_on_overflow=0
net.ipv4.tcp_synack_retries=2
net.ipv4.tcp_syn_retries=3
fs.file-max=1000000
EOF
sysctl --system || true

echo "* - nofile 1048576" | tee -a /etc/security/limits.conf

dnf -y makecache || true
dnf -y install nginx || true

# Global NGINX tuning
cat >/etc/nginx/nginx.conf <<'NGINX'
user  nginx;
worker_processes auto;
worker_rlimit_nofile 1048576;

events { worker_connections 131072; multi_accept on; accept_mutex off; }

http {
    include /etc/nginx/mime.types;
    default_type application/octet-stream;

    # I/O and TCP toggles
    sendfile on; tcp_nopush on; tcp_nodelay on;

    # Short timeouts (CPS mode uses Connection: close)
    keepalive_timeout 15;

    # Tighten idle/slow clients; reset timed out connections early for CPS
    client_body_timeout 10; client_header_timeout 10; send_timeout 10;
    reset_timedout_connection on;

    # Large keepalive request cap (harmless for CPS; useful if switching to keepalive)
    keepalive_requests 100000;

    server_tokens off; access_log off;

    include /etc/nginx/conf.d/*.conf;
}
NGINX

# Pre-create payload files for throughput
__DD_PAYLOAD_BLOCK__

# NGINX server: HTTP:80 with PROXY protocol v2 expected from LB
cat >/etc/nginx/conf.d/freewheel.conf <<'EOF'
server {
    listen 80 proxy_protocol reuseport backlog=262144 fastopen=512;
    server_name _;
    access_log off;

    # Trust the LB subnet (adjust if your LB subnet differs)
    set_real_ip_from 10.0.1.0/24;
    real_ip_header proxy_protocol;
    real_ip_recursive on;

    location = /healthz { return 200 "ok\n"; }
    location /payload_ { root /usr/share/nginx/html; }
    location /          { return 200 "ok\n"; }
}
EOF

rm -f /etc/nginx/conf.d/default.conf
nginx -t && systemctl enable --now nginx
"""

backend_cloud_init = backend_cloud_init_template.replace(
    "__DD_PAYLOAD_BLOCK__", _dd_payload_block
)

# Generator cloud-init: multi-capable locustfile (targets fan out across VIPs when set)
generator_cloud_init = r"""#!/bin/bash
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi
cat <<EOF >/etc/sysctl.d/99-freewheel.conf
net.core.somaxconn=65535
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
fs.file-max=1000000
EOF
sysctl --system || true
echo "* - nofile 1048576" >> /etc/security/limits.conf

dnf -y makecache || true
dnf -y install python3 python3-pip tmux curl || true
pip3 install --no-cache-dir --upgrade pip || true
pip3 install --no-cache-dir locust || true

mkdir -p /home/opc/locustwork/results
chown -R opc:opc /home/opc/locustwork

cat > /home/opc/locustwork/locustfile.py <<'PY'
import os
from itertools import cycle
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
RAW_TARGETS       = os.environ.get("LOCUST_TARGETS", "")
TARGETS           = [t.strip() for t in RAW_TARGETS.split(",") if t.strip()]
NAME_BY_VIP       = os.environ.get("LOCUST_NAME_BY_VIP", "true").lower() == "true"  # default true for per‑VIP labels

class CpsUser(HttpUser):
    host = os.environ.get("LOCUST_DEFAULT_HOST", "https://127.0.0.1")
    wait_time = constant(WAIT_TIME_S)

    def on_start(self):
        self._targets = cycle(TARGETS) if TARGETS else None

    def _next_base(self):
        if self._targets:
            return next(self._targets)
        return self.host

    @task
    def do_request(self):
        path = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        base = self._next_base()
        url  = f"{base}{path}" if base else path
        headers = {"Connection": "close"} if MODE == "cps" else {}
        name_val = path
        if NAME_BY_VIP and base:
            vip = base.replace("https://", "").replace("http://", "")
            name_val = f"{vip}{path}"
        self.client.get(url, headers=headers, verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=name_val)
PY

chown -R opc:opc /home/opc/locustwork
echo READY
"""

with open("cloud-init/backend.sh.tftpl", "w") as f:
    f.write(backend_cloud_init)
with open("cloud-init/generator.sh", "w") as f:
    f.write(generator_cloud_init)

print(
    "Cell 4 complete: Wrote backend.sh.tftpl (HTTP:80 + PPv2, tuned) and generator.sh (Locust multi-target capable; agent enabled; payloads baked)."
)
print(
    "NEXT: Run Cell 5 to generate Terraform (combined Single/Multi with stateless NSGs and IPv6 dual-stack), then Cell 6 tfvars, and Cell 7 apply."
)


In [ ]:
# Cell 5 — Objective: Terraform (combined Single/Multi) using NSGs, VCN/subnets, private flexible LB(s) with TCP listener + TLS offload + PPv2
# - Dual-stack VCN/subnets always enabled
# - When ENABLE_IPV6_FRONTEND is true:
#   * Attempt IPv6 VIP for the LB(s) (ip_mode + ipv6subnet_cidr; support may vary by region/tenancy)
#   * Assign one IPv6 to each generator’s primary VNIC via oci_core_ipv6
# - All tcp_options nested blocks are multi-line per HCL requirements
# - All NSG rules are stateless and mirror previously working rules (including return paths)
# - Default Security List includes IPv6 allow-all (stateless) for ingress/egress

import os

single_ad = (AD_A == AD_B) or (str(AD_B or "").strip() == "")

backend_shape_config_block = ""
if (BACKEND_SHAPE or "").endswith(".Flex"):
    backend_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (
        BACKEND_OCPUS,
        BACKEND_MEMORY_GB,
    )

generator_shape_config_block = ""
if (GENERATOR_SHAPE or "").endswith(".Flex"):
    generator_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (
        GENERATOR_OCPUS,
        GENERATOR_MEMORY_GB,
    )

_enable_v6 = os.environ.get("ENABLE_IPV6_FRONTEND", "true").lower() == "true"

lb_ipv6_block = (
    """
  ip_mode         = "IPV6"
  ipv6subnet_cidr = local.lb_priv_ipv6
"""
    if _enable_v6
    else ""
)

GEN_V6_COUNT_STR = "var.generator_count" if _enable_v6 else "0"

terraform_config_template = """terraform {
  required_providers {
    oci = {
      source  = "oracle/oci"
      version = ">= 7.0.0"
    }
  }
}

provider "oci" {
  config_file_profile = var.oci_profile
  region              = var.region
}

# Variables
variable "oci_profile" {}
variable "region" {}
variable "compartment_id" {}
variable "ad_a" {}
variable "ad_b" {}
variable "image_id" {}
variable "ssh_public_key_content" {}
variable "ssh_private_key_path" {}

variable "backend_count" {}
variable "backend_shape" {}
variable "generator_count" {}
variable "generator_shape" {}

variable "ssh_allowed_cidr" {}
variable "ssh_allowed_v6_cidr" {
  type    = string
  default = "::/0"
}

# TLS PEMs for LB offload (Cell 6 embeds contents)
variable "lb_cert_pem" { type = string }
variable "lb_key_pem"  { type = string }
variable "lb_ca_pem"   { type = string }

variable "lb_min_mbps" {}
variable "lb_max_mbps" {}
variable "lb_count"    {}

variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}

# VCN (dual-stack enabled)
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.0.0.0/16"
  compartment_id = var.compartment_id
  display_name   = "cps-combined-vcn"
  is_ipv6enabled = true
}

# Derive /64s from Oracle-assigned /56
locals {
  vcn_ipv6_base      = oci_core_vcn.vcn.ipv6cidr_blocks[0]
  lb_priv_ipv6       = cidrsubnet(local.vcn_ipv6_base, 8, 1)
  backends_priv_ipv6 = cidrsubnet(local.vcn_ipv6_base, 8, 2)
  gens_pub_ipv6      = cidrsubnet(local.vcn_ipv6_base, 8, 3)
}

# Default Security List managed to stateless allow-all (IPv4 + IPv6)
resource "oci_core_default_security_list" "default_sl" {
  manage_default_resource_id = oci_core_vcn.vcn.default_security_list_id

  # Egress IPv4
  egress_security_rules {
    protocol         = "all"
    destination      = "0.0.0.0/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }
  # Egress IPv6
  egress_security_rules {
    protocol         = "all"
    destination      = "::/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  # Ingress IPv4
  ingress_security_rules {
    protocol    = "all"
    source      = "0.0.0.0/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
  # Ingress IPv6
  ingress_security_rules {
    protocol    = "all"
    source      = "::/0"
    source_type = "CIDR_BLOCK"
    stateless   = true
  }
}

# Gateways & Routes
resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-combined-igw"
}
resource "oci_core_nat_gateway" "nat" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-combined-nat"
}
resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-combined-rt-public"
  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
  route_rules {
    destination       = "::/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}
resource "oci_core_route_table" "rt_private" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-combined-rt-private"
  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_nat_gateway.nat.id
  }
}

# NSGs
resource "oci_core_network_security_group" "nsg_lb" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-lb"
}
resource "oci_core_network_security_group" "nsg_backends" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-backends"
}
resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-generators"
}

# NSG Rules (stateless + symmetric return-path) — ALL tcp_options nested blocks are multi-line
resource "oci_core_network_security_group_security_rule" "lb_ingress_443_from_generators" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}
resource "oci_core_network_security_group_security_rule" "lb_egress_80_to_backends" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_backends.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}
resource "oci_core_network_security_group_security_rule" "lb_ingress_from_backends_src80" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_backends.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 80
      max = 80
    }
  }
}
resource "oci_core_network_security_group_security_rule" "lb_egress_to_generators_src443" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
}
resource "oci_core_network_security_group_security_rule" "backends_ingress_80_from_lb" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}
resource "oci_core_network_security_group_security_rule" "backends_egress_all" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}
# SSH IPv4 ingress
resource "oci_core_network_security_group_security_rule" "gens_ingress_ssh" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ssh_allowed_cidr
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}
# SSH IPv6 ingress
resource "oci_core_network_security_group_security_rule" "gens_ingress_ssh_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ssh_allowed_v6_cidr
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}
# Locust control ports (5557–5558)
resource "oci_core_network_security_group_security_rule" "gens_ingress_locust_master" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 5557
      max = 5558
    }
  }
}
resource "oci_core_network_security_group_security_rule" "gens_ingress_from_generators_src5557_5558" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 5557
      max = 5558
    }
  }
  description = "Stateless return path for Locust ports 5557-5558"
}
# Egress
resource "oci_core_network_security_group_security_rule" "gens_egress_all_v4" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}
resource "oci_core_network_security_group_security_rule" "gens_egress_all_v6" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "::/0"
  stateless                 = true
}
# Return path for LB responses to clients (src 443)
resource "oci_core_network_security_group_security_rule" "gens_ingress_from_lb_src443" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
  description = "Stateless return path for LB responses (source port 443) to generator clients"
}

# Subnets (dual-stack)
resource "oci_core_subnet" "lb_priv" {
  cidr_block                 = "10.0.1.0/24"
  ipv6cidr_block             = local.lb_priv_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "cps-combined-lb-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}
resource "oci_core_subnet" "backends_priv" {
  cidr_block                 = "10.0.2.0/24"
  ipv6cidr_block             = local.backends_priv_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "cps-combined-backends-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}
resource "oci_core_subnet" "gens_pub" {
  cidr_block                 = "10.0.3.0/24"
  ipv6cidr_block             = local.gens_pub_ipv6
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "cps-combined-generators-pub"
  route_table_id             = oci_core_route_table.rt_public.id
  prohibit_public_ip_on_vnic = false
}

# Backends
resource "oci_core_instance" "backend" {
  count               = var.backend_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.backend_shape
__BACKEND_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }
  create_vnic_details {
    subnet_id        = oci_core_subnet.backends_priv.id
    assign_public_ip = false
    nsg_ids          = [oci_core_network_security_group.nsg_backends.id]
  }
  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false
    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }
  display_name = "cps-backend-${count.index}"
  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = base64encode(file("${path.module}/cloud-init/backend.sh.tftpl"))
  }
}

# Generators (IPv6 added via oci_core_ipv6 when toggle is ON)
resource "oci_core_instance" "generator" {
  count               = var.generator_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape
__GENERATOR_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }
  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [oci_core_network_security_group.nsg_generators.id]
  }
  display_name = "cps-gen-${count.index}"
  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = filebase64("${path.module}/cloud-init/generator.sh")
  }
}

data "oci_core_vnic_attachments" "gen_vnic" {
  count          = __GEN_V6_COUNT__
  compartment_id = var.compartment_id
  instance_id    = oci_core_instance.generator[count.index].id
}

resource "oci_core_ipv6" "gen_v6" {
  count     = __GEN_V6_COUNT__
  subnet_id = oci_core_subnet.gens_pub.id
  vnic_id   = data.oci_core_vnic_attachments.gen_vnic[count.index].vnic_attachments[0].vnic_id
}

# Flexible LBs (private) — IPv6 VIP when toggle is ON
resource "oci_load_balancer_load_balancer" "lb" {
  count          = var.lb_count
  compartment_id = var.compartment_id
  display_name   = "cps-flex-lb-${count.index}"
  shape          = "flexible"
  is_private     = true
  subnet_ids     = [oci_core_subnet.lb_priv.id]
  network_security_group_ids = [oci_core_network_security_group.nsg_lb.id]
__LB_IPV6_BLOCK__
  shape_details {
    minimum_bandwidth_in_mbps = var.lb_min_mbps
    maximum_bandwidth_in_mbps = var.lb_max_mbps
  }
}

resource "oci_load_balancer_certificate" "lb_cert" {
  count              = var.lb_count
  load_balancer_id   = element(oci_load_balancer_load_balancer.lb.*.id, count.index)
  certificate_name   = "lb-cert-${count.index}"
  public_certificate = var.lb_cert_pem
  private_key        = var.lb_key_pem
  ca_certificate     = var.lb_ca_pem
}

resource "oci_load_balancer_backend_set" "bs" {
  count            = var.lb_count
  load_balancer_id = element(oci_load_balancer_load_balancer.lb.*.id, count.index)
  name             = "backendset1"
  policy           = "ROUND_ROBIN"
  health_checker {
    protocol          = "TCP"
    port              = 80
    retries           = 3
    interval_ms       = 10000
    timeout_in_millis = 5000
  }
}

resource "oci_load_balancer_backend" "backends" {
  count            = var.lb_count * var.backend_count
  load_balancer_id = element(oci_load_balancer_load_balancer.lb.*.id, floor(count.index / var.backend_count))
  backendset_name  = element(oci_load_balancer_backend_set.bs.*.name, floor(count.index / var.backend_count))
  ip_address       = element(oci_core_instance.backend.*.private_ip, count.index % var.backend_count)
  port             = 80
  weight           = 1
}

resource "oci_load_balancer_listener" "tcp443" {
  count                    = var.lb_count
  load_balancer_id         = element(oci_load_balancer_load_balancer.lb.*.id, count.index)
  name                     = "tcp-443"
  port                     = 443
  protocol                 = "TCP"
  default_backend_set_name = element(oci_load_balancer_backend_set.bs.*.name, count.index)

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.lb_cert[count.index].certificate_name
    verify_peer_certificate = false
    protocols               = ["TLSv1.3", "TLSv1.2"]
    server_order_preference = "ENABLED"
    cipher_suite_name       = "oci-modern-ssl-cipher-suite-v1"
  }

  connection_configuration {
    idle_timeout_in_seconds            = 1200
    backend_tcp_proxy_protocol_version = 2
  }

  depends_on = [oci_load_balancer_certificate.lb_cert]
}

# Outputs (single-friendly + multi-friendly)
output "lb_id"                 { value = element(oci_load_balancer_load_balancer.lb.*.id, 0) }
output "lb_ip_addresses"       { value = [for d in oci_load_balancer_load_balancer.lb[0].ip_address_details : d.ip_address] }
output "lb_ids"                { value = [for lb in oci_load_balancer_load_balancer.lb : lb.id] }
output "lb_ip_addresses_multi" { value = [for lb in oci_load_balancer_load_balancer.lb : [for d in lb.ip_address_details : d.ip_address]] }
output "lb_ip_addresses_flat"  { value = flatten([for lb in oci_load_balancer_load_balancer.lb : [for d in lb.ip_address_details : d.ip_address]]) }
output "gen_public_ips"        { value = [for i in oci_core_instance.generator : i.public_ip] }
output "backend_private_ips"   { value = [for i in oci_core_instance.backend   : i.private_ip] }
output "gen_ipv6_ips"          { value = oci_core_ipv6.gen_v6[*].ip_address }
"""

terraform_config = (
    terraform_config_template.replace(
        "__BACKEND_SHAPE_CONFIG__", backend_shape_config_block
    )
    .replace("__GENERATOR_SHAPE_CONFIG__", generator_shape_config_block)
    .replace("__LB_IPV6_BLOCK__", lb_ipv6_block)
    .replace("__GEN_V6_COUNT__", GEN_V6_COUNT_STR)
)

with open("main.tf", "w") as f:
    f.write(terraform_config)

print("Cell 5 complete: Terraform main.tf written (combined Single/Multi; stateless NSGs; dual-stack; PPv2; TLS 1.3+1.2).")
print("NEXT: Run Cell 6 to write terraform.tfvars, then Cell 7 to apply.")


In [ ]:
# Cell 6 — Objective: Write terraform.tfvars using selections (supports Single/Multi), includes LB PEMs and IPv6 SSH CIDR

import os


def _read_text_required(path: str, label: str) -> str:
    p = os.path.expanduser(path or "")
    if not p or not os.path.exists(p):
        raise ValueError(f"{label} missing. Set via Cell 3. Current: {path or '(unset)'}")
    with open(p, "r") as f:
        return f.read().strip()


def _read_text_optional(path: str) -> str:
    p = os.path.expanduser(path or "")
    if not p or not os.path.exists(p):
        return ""
    with open(p, "r") as f:
        return f.read().strip()

# Validate required selections from Cell 3
if not IMAGE_ID:
    raise ValueError("IMAGE_ID is not set. Run Cell 3 and select an Image.")
if not BACKEND_SHAPE:
    raise ValueError("BACKEND_SHAPE is not set. Run Cell 3 and select a Backend Shape.")
if not GENERATOR_SHAPE:
    raise ValueError("GENERATOR_SHAPE is not set. Run Cell 3 and select a Generator Shape.")

if not SSH_PUBLIC_KEY_PATH or not os.path.exists(os.path.expanduser(SSH_PUBLIC_KEY_PATH)):
    raise ValueError(f"SSH public key missing. Pick a valid key in Cell 3. Current: {SSH_PUBLIC_KEY_PATH}")
if not SSH_PRIVATE_KEY_PATH or not os.path.exists(os.path.expanduser(SSH_PRIVATE_KEY_PATH)):
    raise ValueError(f"SSH private key missing. Pick a valid key in Cell 3. Current: {SSH_PRIVATE_KEY_PATH}")

# Load SSH public key content
with open(os.path.expanduser(SSH_PUBLIC_KEY_PATH), "r") as f:
    ssh_public_key_content = f.read().strip()
ssh_public_key_content_escaped = ssh_public_key_content.replace("\"", "\\\"")

# PEMs (paths were persisted in Cell 3 env)
LB_CERT_PEM_PATH = os.path.expanduser(os.environ.get("LB_CERT_PEM_PATH", ""))
LB_KEY_PEM_PATH  = os.path.expanduser(os.environ.get("LB_KEY_PEM_PATH",  ""))
LB_CA_PEM_PATH   = os.path.expanduser(os.environ.get("LB_CA_PEM_PATH",   ""))

lb_cert_pem = _read_text_required(LB_CERT_PEM_PATH, "LB cert PEM")
lb_key_pem  = _read_text_required(LB_KEY_PEM_PATH,  "LB key PEM")
lb_ca_pem   = _read_text_optional(LB_CA_PEM_PATH)

# IPv6 SSH allowed CIDR (override via env SSH_ALLOWED_V6_CIDR; default ::/0)
SSH_ALLOWED_V6_CIDR = os.environ.get("SSH_ALLOWED_V6_CIDR", "::/0").strip()

# Topology (LB count)
LB_TOPOLOGY_ENV = os.environ.get("LB_TOPOLOGY", "single").strip().lower()
LB_COUNT_ENV = int(os.environ.get("LB_COUNT", "1"))
LB_COUNT_EFFECTIVE = 1 if LB_TOPOLOGY_ENV == "single" else max(1, LB_COUNT_ENV)

# Write tfvars matching main.tf variables exactly
# main.tf variables: oci_profile, region, compartment_id, ad_a, ad_b, image_id,
# ssh_public_key_content, ssh_private_key_path, backend_count, backend_shape,
# generator_count, generator_shape, ssh_allowed_cidr, ssh_allowed_v6_cidr,
# lb_min_mbps, lb_max_mbps, lb_count, lb_cert_pem, lb_key_pem, lb_ca_pem,
# (bastion_plugin_name has a default, but we include it for clarity)

ssh_private_key_abs = os.path.expanduser(SSH_PRIVATE_KEY_PATH)

tfvars = f"""
oci_profile            = "{OCI_PROFILE}"
region                 = "{REGION}"
compartment_id         = "{COMPARTMENT_ID}"
ad_a                   = "{AD_A}"
ad_b                   = "{AD_B}"
image_id               = "{IMAGE_ID}"
ssh_public_key_content = "{ssh_public_key_content_escaped}"
ssh_private_key_path   = "{ssh_private_key_abs}"

backend_count   = {int(BACKEND_COUNT)}
backend_shape   = "{BACKEND_SHAPE}"

generator_count = {int(GENERATOR_COUNT)}
generator_shape = "{GENERATOR_SHAPE}"

ssh_allowed_cidr    = "{SSH_ALLOWED_CIDR}"
ssh_allowed_v6_cidr = "{SSH_ALLOWED_V6_CIDR}"

lb_min_mbps = {int(LB_MIN_MBPS)}
lb_max_mbps = {int(LB_MAX_MBPS)}
lb_count    = {int(LB_COUNT_EFFECTIVE)}

lb_cert_pem = <<EOCERT
{lb_cert_pem}
EOCERT

lb_key_pem  = <<EOKEY
{lb_key_pem}
EOKEY

lb_ca_pem   = <<EOCA
{lb_ca_pem}
EOCA

bastion_plugin_name = "Bastion"
""".lstrip()

with open("terraform.tfvars", "w") as f:
    f.write(tfvars)

print("Cell 6 complete: terraform.tfvars written (includes lb_count and ssh_allowed_v6_cidr).")
print(f"  LB_TOPOLOGY={LB_TOPOLOGY_ENV} | LB_COUNT_EFFECTIVE={LB_COUNT_EFFECTIVE}")
print(f"  SSH_ALLOWED_V6_CIDR={SSH_ALLOWED_V6_CIDR}")
print("NEXT: Run Cell 7 to terraform init/apply, then continue with Cell 8.")


In [ ]:
# Cell 7 — Objective: Initialize and apply Terraform (fresh build)

# This cell assumes main.tf and terraform.tfvars are in the current working directory
# (the same folder as this notebook). It performs a non-interactive apply.

!terraform init
!terraform apply -auto-approve -var-file=terraform.tfvars

print("Cell 7 complete: Terraform apply finished.")
print("NEXT: Run Cell 8 to capture outputs (LBs/Generators/Backends) and persist for downstream cells.")


In [ ]:
# Cell 8 — Objective: Extract LB/instance outputs and persist them for downstream cells (IPv6-aware, Single/Multi)

import os, json, subprocess


def _tf_output_json() -> dict:
    r = subprocess.run(["terraform", "output", "-json"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"terraform output failed:\n{r.stderr}")
    try:
        return json.loads(r.stdout or "{}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Could not parse terraform output JSON: {e}")


def _val(outputs: dict, key: str):
    obj = outputs.get(key, {})
    return obj.get("value", obj if obj else None)


def _is_ipv6(addr: str) -> bool:
    return ":" in (addr or "")


def _bracket_if_ipv6(host: str) -> str:
    return (f"[{host}]" if _is_ipv6(host) and not (host.startswith("[") and host.endswith("]")) else host)


outputs = _tf_output_json()

# Single-friendly
LB_ID  = _val(outputs, "lb_id")
LB_IPS_SINGLE = _val(outputs, "lb_ip_addresses") or []

# Multi-friendly
LB_IDS          = _val(outputs, "lb_ids") or []
LB_IPS_MULTI    = _val(outputs, "lb_ip_addresses_multi") or []  # list of lists
LB_IPS_FLAT     = _val(outputs, "lb_ip_addresses_flat") or []

# Instances
GEN_IPS_V4 = _val(outputs, "gen_public_ips") or []
GEN_IPS_V6 = _val(outputs, "gen_ipv6_ips") or []
BACKEND_IPS = _val(outputs, "backend_private_ips") or []

# Determine topology and preference
LB_TOPOLOGY = (os.environ.get("LB_TOPOLOGY", "single").strip().lower())
prefer_v6 = os.environ.get("ENABLE_IPV6_FRONTEND", "true").strip().lower() in ("1","true","yes","y")

# Compute TARGET_HOST
candidates = []
if LB_TOPOLOGY == "single":
    candidates = LB_IPS_SINGLE or []
else:
    candidates = LB_IPS_FLAT or []

v6_list = [ip for ip in candidates if _is_ipv6(ip)]
v4_list = [ip for ip in candidates if not _is_ipv6(ip)]

TARGET_HOST_RAW = None
WARN = None
if prefer_v6:
    if v6_list:
        TARGET_HOST_RAW = v6_list[0]
    elif v4_list:
        TARGET_HOST_RAW = v4_list[0]
        WARN = "IPv6 frontend preferred but no IPv6 VIP found; using IPv4 VIP fallback."
    elif candidates:
        TARGET_HOST_RAW = candidates[0]
else:
    if v4_list:
        TARGET_HOST_RAW = v4_list[0]
    elif candidates:
        TARGET_HOST_RAW = candidates[0]
        WARN = "IPv4 preferred but only IPv6 VIP(s) present; using IPv6 VIP."

TARGET_HOST = _bracket_if_ipv6(TARGET_HOST_RAW) if TARGET_HOST_RAW else None
TARGET_URL = f"https://{TARGET_HOST}" if TARGET_HOST else None

# Persist for later cells and external tools (best‑effort)
os.environ["LB_ID"] = str(LB_ID or "")
os.environ["TARGET_HOST"] = str(TARGET_HOST or "")
os.environ["TARGET_URL"] = str(TARGET_URL or "")

# For SSH helpers (Cell 9)
os.environ["GEN_IPS_V4_JSON"] = json.dumps(GEN_IPS_V4)
os.environ["GEN_IPS_V6_JSON"] = json.dumps(GEN_IPS_V6)

# Multi-specific env for orchestration
if LB_IPS_FLAT:
    os.environ["LB_IPS_FLAT_JSON"] = json.dumps(LB_IPS_FLAT)
if LB_IDS:
    os.environ["LB_IDS_JSON"] = json.dumps(LB_IDS)

# Save a snapshot of outputs
OUTPUT_DIR = globals().get("OUTPUT_DIR", "./results")
TS_UTC = globals().get("TS_UTC", "") or ""
os.makedirs(OUTPUT_DIR, exist_ok=True)

snap = {
    "lb_topology": LB_TOPOLOGY,
    "lb_id": LB_ID,
    "lb_ids": LB_IDS,
    "lb_ips_single": LB_IPS_SINGLE,
    "lb_ips_multi": LB_IPS_MULTI,
    "lb_ips_flat": LB_IPS_FLAT,
    "gen_public_ips_v4": GEN_IPS_V4,
    "gen_ipv6_ips": GEN_IPS_V6,
    "backend_private_ips": BACKEND_IPS,
    "target_host_raw": TARGET_HOST_RAW,
    "target_host": TARGET_HOST,
    "target_url": TARGET_URL,
    "prefer_v6": prefer_v6,
}

snap_name = f"terraform_outputs_{TS_UTC or 'latest'}.json"
snap_path = os.path.join(OUTPUT_DIR, snap_name)
with open(snap_path, "w") as f:
    json.dump(snap, f, indent=2)

# Print concise, topology-relevant summary
if LB_TOPOLOGY == "single":
    print("LB_ID:", LB_ID)
    print("LB_IPS (single):", LB_IPS_SINGLE)
else:
    print("LB_IDS:", LB_IDS)
    print("LB_IPS_FLAT:", LB_IPS_FLAT)

print("Generators (IPv4):", GEN_IPS_V4)
print("Generators (IPv6):", GEN_IPS_V6)
print("Backends:", BACKEND_IPS)
print("TARGET_HOST (VIP):", TARGET_HOST)
print("Saved outputs snapshot:", snap_path)
if WARN:
    print("Note:", WARN)

print("Cell 8 complete: Outputs captured and persisted (IPv6-aware, topology-aware).")
print("NEXT: Run Cell 9 for SSH + Monitoring helpers.")


In [ ]:
# Cell 9 — Objective: SSH helpers, LB bandwidth update, Monitoring helpers (dual-stack aware, SSH key consistent)

import os, time, json
import paramiko
import pandas as pd
from datetime import datetime, timezone, timedelta
import oci

SSH_KEY_PASSPHRASE = os.environ.get("SSH_KEY_PASSPHRASE", None)
LB_METRICS_NAMESPACE = "oci_lbaas"

# Resolve generator IPs (prefer IPv6 for SSH when enabled and available)
USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")

try:
    GEN_IPS_V4 = json.loads(os.environ.get("GEN_IPS_V4_JSON", "[]"))
    if not isinstance(GEN_IPS_V4, list): GEN_IPS_V4 = []
except Exception:
    GEN_IPS_V4 = []
try:
    GEN_IPS_V6 = json.loads(os.environ.get("GEN_IPS_V6_JSON", "[]"))
    if not isinstance(GEN_IPS_V6, list): GEN_IPS_V6 = []
except Exception:
    GEN_IPS_V6 = []

if USE_SSH_IPV6 and GEN_IPS_V6:
    GEN_IPS = GEN_IPS_V6; SSH_AF = "IPv6"
elif GEN_IPS_V4:
    GEN_IPS = GEN_IPS_V4; SSH_AF = "IPv4"
else:
    GEN_IPS = globals().get("GEN_IPS", []) or []; SSH_AF = "fallback"

if not GEN_IPS:
    raise RuntimeError("No generator hosts available for SSH. Ensure Cell 7 applied and Cell 8 ran successfully.")

# SSH key resolution — consistent across cells

def _ssh_key_path():
    return os.path.expanduser((globals().get("SSH_PRIVATE_KEY_PATH") or os.environ.get("SSH_PRIVATE_KEY_PATH") or "~/.ssh/id_rsa"))

# --- SSH helpers ---

def _load_pkey(key_path: str, passphrase: str | None):
    kpath = os.path.expanduser(key_path)
    if not os.path.exists(kpath):
        raise FileNotFoundError(f"SSH private key not found: {kpath}")
    excs = []
    for KeyClass in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return KeyClass.from_private_key_file(kpath, password=passphrase)
        except Exception as e:
            excs.append(repr(e))
    raise paramiko.SSHException("Could not load SSH private key. Path: %s\n - %s" % (kpath, "\n - ".join(excs)))


def ssh_exec(host, user="opc", key_path=None, command="echo ok", timeout=600):
    pkey = _load_pkey(key_path or _ssh_key_path(), SSH_KEY_PASSPHRASE)
    ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username=user, pkey=pkey, timeout=30)
    stdin, stdout, stderr = ssh.exec_command(command, timeout=timeout)
    try: out = stdout.read().decode("utf-8", errors="ignore")
    except Exception: out = ""
    try: err = stderr.read().decode("utf-8", errors="ignore")
    except Exception: err = ""
    try: rc = stdout.channel.recv_exit_status()
    except Exception: rc = None
    ssh.close(); return out.strip(), err.strip(), rc


def ssh_exec_quick(host, command):
    pkey = _load_pkey(_ssh_key_path(), SSH_KEY_PASSPHRASE)
    ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    transport = ssh.get_transport(); chan = transport.open_session(); chan.exec_command(command)
    time.sleep(0.5)
    try: chan.close()
    except Exception: pass
    ssh.close()


def ssh_exec_many(hosts, cmd, timeout=1200):
    return {h: ssh_exec(h, command=cmd, timeout=timeout) for h in (hosts or [])}

# --- OCI monitoring clients ---
OCI_CONFIG_FILE = os.environ.get("OCI_CONFIG_FILE", "~/.oci/config")
OCI_PROFILE     = os.environ.get("OCI_PROFILE", "DEFAULT")
REGION          = os.environ.get("REGION", "")
LB_ID           = os.environ.get("LB_ID", "")

cfg = oci.config.from_file(os.path.expanduser(OCI_CONFIG_FILE), OCI_PROFILE)
cfg["region"] = REGION
lbc = oci.load_balancer.LoadBalancerClient(cfg)
mon = oci.monitoring.MonitoringClient(cfg)

# Best-effort compartment resolution for metrics
try:
    _lb = lbc.get_load_balancer(LB_ID).data if LB_ID else None
    COMPARTMENT_ID = (getattr(_lb, "compartment_id", os.environ.get("COMPARTMENT_ID", "")) or "")
except Exception:
    COMPARTMENT_ID = os.environ.get("COMPARTMENT_ID", "")

# Monitoring helpers

def list_available_lb_metrics(lb_id: str):
    try:
        details = oci.monitoring.models.ListMetricsDetails(namespace=LB_METRICS_NAMESPACE, group_by=["name"], dimension_filters={"resourceId": lb_id})
        resp = mon.list_metrics(compartment_id=COMPARTMENT_ID, list_metrics_details=details)
        names = sorted({m.name for m in resp.data if getattr(m, "name", None)})
        if names: return names
    except Exception as e:
        print("list_metrics failed; using fallback set. Reason:", repr(e))
    return sorted(list({"NewConnections","AcceptedConnections","HandledConnections","ActiveConnections","ClosedConnections","BytesIn","BytesOut","BytesReceived","BytesSent"}))


def get_lb_metric_df(lb_id: str, metric_name: str, start_iso: str, end_iso: str, resolution="1m", agg="sum"):
    fn = {"sum": "sum()", "mean": "mean()", "max": "max()", "min": "min()"}.get(agg, "sum()")
    lb_id_escaped = (lb_id or "").replace('"', '\\"')
    query = f'{metric_name}[{resolution}]{{resourceId = "{lb_id_escaped}"}}.{fn}'
    details = oci.monitoring.models.SummarizeMetricsDataDetails(namespace=LB_METRICS_NAMESPACE, query=query, start_time=start_iso, end_time=end_iso, resolution=resolution)
    resp = mon.summarize_metrics_data(compartment_id=COMPARTMENT_ID, summarize_metrics_data_details=details)
    rows = []
    for item in resp.data or []:
        for d in item.aggregated_datapoints or []:
            rows.append({"timestamp": d.timestamp, "value": d.value})
    return (pd.DataFrame(rows).sort_values("timestamp") if rows else pd.DataFrame(columns=["timestamp","value"]))


def update_lb_bandwidth(lb_id: str, mbps: int, wait=True):
    details = oci.load_balancer.models.UpdateLoadBalancerShapeDetails(minimum_bandwidth_in_mbps=mbps, maximum_bandwidth_in_mbps=mbps)
    lbc.update_load_balancer_shape(lb_id, details)
    if wait:
        for _ in range(90):
            lb = lbc.get_load_balancer(lb_id).data
            if getattr(lb, "lifecycle_state", None) == "ACTIVE": break
            time.sleep(5)
    print(f"LB cap set to {mbps} Mbps")


def backend_set_health(lb_id: str, backendset_name="backendset1"):
    return lbc.get_backend_set_health(lb_id, backendset_name).data

# Summary & quick reachability test
print(f"SSH key: {_ssh_key_path()}")
print(f"Using {SSH_AF} for SSH to generators; hosts: {GEN_IPS}")

ok = 0
for h in GEN_IPS:
    try:
        out, err, rc = ssh_exec(h, command="echo ok", timeout=10)
        status = (out or err or "").strip() or f"rc={rc}"
        print(f"{h}: {status}")
        if rc == 0: ok += 1
    except Exception as e:
        print(f"{h}: SSH failed: {e}")

print(f"Cell 9 complete: SSH + Monitoring helpers loaded. {ok}/{len(GEN_IPS)} hosts reachable.")
print("NEXT: Run Cell 10 for generator→LB HTTPS sanity checks (IPv6-aware, Multi aware).")


In [ ]:
# Cell 10 — Objective: Sanity-check HTTPS to LB (IPv6-aware; Single/Multi aware)
# - Single: test TARGET_HOST only
# - Multi: test each VIP in LB_IPS_FLAT from each generator

import os, json
from typing import Tuple

# From prior cells:
# - GEN_IPS (Cell 9)
# - TARGET_HOST, LB_IPS_FLAT_JSON (Cell 8)
# - ssh_exec(host, command, timeout=..., user=...) (Cell 9)

# Health path (defined in Cell 2). Fail-fast if missing.
HEALTH_ENDPOINT_PATH = os.environ.get("HEALTH_ENDPOINT_PATH") or (
    globals().get("HEALTH_ENDPOINT_PATH") if "HEALTH_ENDPOINT_PATH" in globals() else None
)
if not HEALTH_ENDPOINT_PATH:
    raise RuntimeError("HEALTH_ENDPOINT_PATH is not set. Define it in Cell 2 (e.g., '/healthz').")

LB_TOPOLOGY = (os.environ.get("LB_TOPOLOGY", "single").strip().lower())

# Determine IPv6 curl flag by whether the host is bracketed

def _use_v6(url_or_host: str) -> bool:
    return "[" in (url_or_host or "") and "]" in (url_or_host or "")

# Run a HEAD request with strict timeouts; ignore cert validation (-k)

def curl_status_from_gen(host: str, url: str, timeout: int = 4) -> Tuple[str, str, int]:
    v6flag = "-6 " if _use_v6(url) else ""
    cmd = f"curl -skI {v6flag}--connect-timeout {timeout} --max-time {timeout} {url} | head -n1 || true"
    return ssh_exec(host, command=cmd, timeout=timeout + 3)

# Build test targets
TEST_TARGETS = []
if LB_TOPOLOGY == "single":
    TARGET_HOST = os.environ.get("TARGET_HOST") or (globals().get("TARGET_HOST") if "TARGET_HOST" in globals() else None)
    if not TARGET_HOST:
        raise RuntimeError("TARGET_HOST is not set. Run Cell 8 to populate TARGET_HOST.")
    TEST_TARGETS = [f"https://{TARGET_HOST}{HEALTH_ENDPOINT_PATH}"]
else:
    try:
        LB_IPS_FLAT = json.loads(os.environ.get("LB_IPS_FLAT_JSON", "[]"))
        if not isinstance(LB_IPS_FLAT, list): LB_IPS_FLAT = []
    except Exception:
        LB_IPS_FLAT = []
    if not LB_IPS_FLAT:
        raise RuntimeError("LB_IPS_FLAT is empty. Run Cell 8 to populate outputs.")
    # dedupe preserving order
    seen = set(); uniq = []
    for ip in LB_IPS_FLAT:
        if ip not in seen:
            seen.add(ip); uniq.append(ip)
    def _bracket_if_ipv6(host: str) -> str:
        return (f"[{host}]" if ":" in (host or "") and not (host.startswith("[") and host.endswith("]")) else host)
    TEST_TARGETS = [f"https://{_bracket_if_ipv6(ip)}{HEALTH_ENDPOINT_PATH}" for ip in uniq]

print("Testing targets:")
for t in TEST_TARGETS:
    print(" -", t)

# Execute from each generator
ok_total = 0; total_checks = 0
for h in GEN_IPS:
    for url in TEST_TARGETS:
        st_out, st_err, rc = curl_status_from_gen(h, url, timeout=4)
        line = (st_out or "").strip() or (st_err or "").strip() or f"rc={rc}"
        print(f"{h} => {url} => {line}")
        total_checks += 1
        if line.startswith("HTTP/") and (" 200 " in line or line.endswith(" 200")):
            ok_total += 1

print(f"Cell 10 complete: {ok_total}/{total_checks} successful HEAD 200 checks.")
print("NEXT: Run Cell 11 to prepare generators (install/verify Locust, write locustfile).")


In [ ]:
# Cell 11 — Objective: Prepare generators seamlessly (verify/install Locust for opc), write locustfile, confirm readiness

import os, shlex, json

# Resolve generator hosts from env if GEN_IPS not in globals
try:
    GEN_IPS  # noqa
except NameError:
    try:
        GEN_IPS_V6 = json.loads(os.environ.get("GEN_IPS_V6_JSON", "[]"))
    except Exception:
        GEN_IPS_V6 = []
    try:
        GEN_IPS_V4 = json.loads(os.environ.get("GEN_IPS_V4_JSON", "[]"))
    except Exception:
        GEN_IPS_V4 = []
    USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")
    GEN_IPS = GEN_IPS_V6 if (USE_SSH_IPV6 and GEN_IPS_V6) else (GEN_IPS_V4 or [])

if not GEN_IPS:
    raise RuntimeError("GEN_IPS not available. Run Cell 9 first or re-run Cell 8 to populate outputs.")

# ssh helpers (fallback if not already available from Cells 9/12)
try:
    ssh_exec  # noqa
except NameError:
    import paramiko
    def _ssh_key_path():
        return os.path.expanduser((globals().get("SSH_PRIVATE_KEY_PATH") or os.environ.get("SSH_PRIVATE_KEY_PATH") or "~/.ssh/id_rsa"))
    def _load_pkey(path, pw):
        path = os.path.expanduser(path)
        for K in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
            try: return K.from_private_key_file(path, password=pw)
            except Exception: pass
        raise paramiko.SSHException("Could not load SSH private key")
    def ssh_exec(host, user="opc", key_path=None, command="echo ok", timeout=120):
        pkey = _load_pkey(key_path or _ssh_key_path(), os.environ.get("SSH_KEY_PASSPHRASE", None))
        ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username=user, pkey=pkey, timeout=30)
        _, stdout, stderr = ssh.exec_command(command, timeout=timeout)
        out = (stdout.read().decode("utf-8", errors="ignore") or "").strip()
        err = (stderr.read().decode("utf-8", errors="ignore") or "").strip()
        try: rc = stdout.channel.recv_exit_status()
        except Exception: rc = None
        ssh.close(); return out, err, rc

LOCUST_WORKDIR = globals().get("LOCUST_WORKDIR") or os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")
if not LOCUST_WORKDIR:
    raise RuntimeError("LOCUST_WORKDIR is not set. Define it in Cell 2.")

# Multi-capable Locustfile (matches generator cloud-init), safe for Single (will use default host if no LOCUST_TARGETS)
LOCUSTFILE_CODE = r"""import os
from itertools import cycle
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
RAW_TARGETS       = os.environ.get("LOCUST_TARGETS", "")
TARGETS           = [t.strip() for t in RAW_TARGETS.split(",") if t.strip()]
NAME_BY_VIP       = os.environ.get("LOCUST_NAME_BY_VIP", "true").lower() == "true"

class CpsUser(HttpUser):
    host = os.environ.get("LOCUST_DEFAULT_HOST", "https://127.0.0.1")
    wait_time = constant(WAIT_TIME_S)

    def on_start(self):
        self._targets = cycle(TARGETS) if TARGETS else None

    def _next_base(self):
        if self._targets:
            return next(self._targets)
        return self.host

    @task
    def do_request(self):
        path = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        base = self._next_base()
        url  = f"{base}{path}" if base else path
        headers = {"Connection": "close"} if MODE == "cps" else {}
        name_val = path
        if NAME_BY_VIP and base:
            vip = base.replace("https://", "").replace("http://", "")
            name_val = f"{vip}{path}"
        self.client.get(url, headers=headers,
                        verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=name_val)"""

verify_cmd = "python3 - <<'PY'\ntry:\n import locust; print('LOCUST_OK')\nexcept Exception as e:\n print('LOCUST_MISSING', e)\nPY"
install_cmd = (
    "bash -lc '"
    "python3 -m pip install --user --upgrade pip setuptools wheel >/dev/null 2>&1 || true; "
    "python3 -m pip show locust >/dev/null 2>&1 || python3 -m pip install --user \"locust>=2.15.1,<3\" >/dev/null 2>&1 || true; "
    "python3 - <<PY\ntry:\n import locust; print(\"LOCUST_OK\")\nexcept Exception as e:\n print(\"LOCUST_MISSING\", e)\nPY'"
)

ok = 0
for host in GEN_IPS:
    out, err, rc = ssh_exec(host, command=verify_cmd, timeout=60)
    if not (out or "").strip().startswith("LOCUST_OK"):
        _o, _e, _r = ssh_exec(host, command=install_cmd, timeout=240)
        out = _o or _e
    if (out or "").strip().startswith("LOCUST_OK"): ok += 1; print(f"{host}: LOCUST_OK")
    else: print(f"{host}: Locust not available — last output: {(out or '').strip()} {(err or '').strip()}")

WDQ = shlex.quote(LOCUST_WORKDIR)
write_locustfile_cmd = f"bash -lc 'mkdir -p {WDQ}/results && cat > {WDQ}/locustfile.py <<\"PY\"\n" + LOCUSTFILE_CODE + "\nPY'"
for host in GEN_IPS:
    wout, werr, wrc = ssh_exec(host, command=write_locustfile_cmd, timeout=60)
    print(f"{host}: locustfile.py ready at {LOCUST_WORKDIR}/locustfile.py (rc={wrc})")

print(f"Cell 11 complete: {ok}/{len(GEN_IPS)} generators have Locust importable; locustfile written.")
print("NEXT: Run Cell 12 to start UI/Workers (Multi-aware; per‑VIP naming enabled).")


In [ ]:
# Cell 12 — Objective: Start Locust UI master + workers (Multi-aware; IPv6 VIP preserved; per‑VIP labels enabled)
# - Multi: exports LOCUST_TARGETS from LB_IPS_FLAT and sets LOCUST_NAME_BY_VIP=true (UI shows per‑VIP rows + Aggregated)
# - Single: passes --host https://{TARGET_HOST}
# - UI will display DEFAULT_HOST_URL derived from the first IPv6 VIP (or first VIP) via LOCUST_DEFAULT_HOST + --host

import os, shlex, json, time, paramiko

UI_WEB_PORT = int(globals().get("UI_WEB_PORT") or os.environ.get("UI_WEB_PORT", "8089"))
LOCUST_WORKDIR = globals().get("LOCUST_WORKDIR") or os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")
TARGET_HOST = (globals().get("TARGET_HOST") or os.environ.get("TARGET_HOST") or "").strip()

LOCUST_VERIFY_TLS    = (globals().get("LOCUST_VERIFY_TLS") if "LOCUST_VERIFY_TLS" in globals() else (os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"))
LOCUST_CONNECT_MS    = int(globals().get("LOCUST_CONNECT_TIMEOUT") or os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))
LOCUST_READ_MS       = int(globals().get("LOCUST_READ_TIMEOUT")   or os.environ.get("LOCUST_READ_TIMEOUT_MS",  "15000"))
LOCUST_WAIT_TIME_SEC = float(globals().get("LOCUST_WAIT_TIME_SEC") or os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
TEST_MODE            = (globals().get("TEST_MODE") or os.environ.get("TEST_MODE", "cps")).lower()
HEALTH_ENDPOINT_PATH = globals().get("HEALTH_ENDPOINT_PATH") or os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = globals().get("THROUGHPUT_ENDPOINT_PATH") or os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")

WORKERS_PER_HOST     = globals().get("WORKERS_PER_HOST", "auto")
CPU_RESERVE          = int(globals().get("CPU_RESERVE", 1))
MIN_WORKERS_PER_HOST = int(globals().get("MIN_WORKERS_PER_HOST", 1))
MAX_WORKERS_PER_HOST = int(globals().get("MAX_WORKERS_PER_HOST", 32))

LB_TOPOLOGY = (os.environ.get("LB_TOPOLOGY", "single").strip().lower())
try:
    LB_IPS_FLAT = json.loads(os.environ.get("LB_IPS_FLAT_JSON", "[]"))
    if not isinstance(LB_IPS_FLAT, list): LB_IPS_FLAT = []
except Exception:
    LB_IPS_FLAT = []

# SSH helpers
try:
    ssh_exec  # noqa
except NameError:
    def _ssh_key_path():
        return os.path.expanduser((globals().get("SSH_PRIVATE_KEY_PATH") or os.environ.get("SSH_PRIVATE_KEY_PATH") or "~/.ssh/id_rsa"))
    SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)
    def _load_pkey(path, pw):
        for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
            try: return Key.from_private_key_file(path, password=pw)
            except Exception: pass
        raise paramiko.SSHException("Could not load SSH private key")
    def ssh_exec(host, user="opc", key_path=None, command="echo ok", timeout=120):
        pkey = _load_pkey(key_path or _ssh_key_path(), SSH_PASS)
        ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username='opc', pkey=pkey, timeout=30)
        _, stdout, stderr = ssh.exec_command(command, timeout=timeout)
        out = (stdout.read().decode('utf-8', errors='ignore') or '').strip()
        err = (stderr.read().decode('utf-8', errors='ignore') or '').strip()
        try: rc = stdout.channel.recv_exit_status()
        except Exception: rc = None
        ssh.close(); return out, err, rc

USE_SSH_IPV6 = os.environ.get("USE_SSH_IPV6", "false").strip().lower() in ("1","true","yes","y")

def _load_json_env(name, default="[]"):
    try:
        v = json.loads(os.environ.get(name, default))
        return v if isinstance(v, list) else []
    except Exception:
        return []

GEN_IPS_V6 = _load_json_env("GEN_IPS_V6_JSON")
GEN_IPS_V4 = _load_json_env("GEN_IPS_V4_JSON")
GEN_IPS = GEN_IPS_V6 if (USE_SSH_IPV6 and GEN_IPS_V6) else (GEN_IPS_V4 or (globals().get("GEN_IPS", []) or []))
if not GEN_IPS:
    raise RuntimeError("No generator hosts available. Run Cells 8–11 first.")
MASTER = GEN_IPS[0]

# Helper: detect master private IPv4 (workers attach via private IP when IPv4 control-plane)

def _detect_private_ipv4(host):
    cmd = r"""bash -lc '
        if curl -fsS -H "Authorization: Bearer Oracle" http://169.254.169.254/opc/v2/vnics/ >/tmp/vn.json 2>/dev/null; then :;
        elif curl -fsS http://169.254.169.254/opc/v1/vnics/ >/tmp/vn.json 2>/dev/null; then :;
        else : > /tmp/vn.json; fi
        ip=$(tr -d "\n" </tmp/vn.json | sed -n 's/.*"privateIp"[[:space:]]*:[[:space:]]*"\([0-9.]*\)".*/\1/p' | head -1)
        if [ -z "$ip" ]; then ip=$(ip -4 route get 1.1.1.1 2>/dev/null | awk '{for(i=1;i<=NF;i++) if($i=="src") print $(i+1)}' | head -1); fi
        if [ -z "$ip" ]; then ip=$(hostname -I 2>/dev/null | tr ' ' '\n' | awk '($1 ~ /^10\./)||($1 ~ /^192\.168\./)||($1 ~ /^172\.(1[6-9]|2[0-9]|3[0-1])\./){print; exit}'); fi
        if [ -z "$ip" ]; then ip=$(ip -4 addr show scope global 2>/dev/null | awk '/inet /{print $2}' | cut -d/ -f1 | awk '($1 ~ /^10\./)||($1 ~ /^192\.168\./)||($1 ~ /^172\.(1[6-9]|2[0-9]|3[0-1])\./){print; exit}'); fi
        echo "$ip'"""
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    return (out or "").strip()

if USE_SSH_IPV6 and ":" in MASTER:
    MASTER_BIND_HOST = "::"
    MASTER_ADDR_FOR_WORKERS = MASTER
else:
    MASTER_BIND_HOST = "0.0.0.0"
    MASTER_ADDR_FOR_WORKERS = _detect_private_ipv4(MASTER) or MASTER
print(f"MASTER={MASTER} | BIND={MASTER_BIND_HOST} | workers-> {MASTER_ADDR_FOR_WORKERS}")

# Ensure remote workspace exists on master
SSH_KEY = os.path.expanduser(os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa"))
SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)

def _load_pkey(path, pw):
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try: return Key.from_private_key_file(path, password=pw)
        except Exception: pass
    raise paramiko.SSHException("Could not load SSH private key")

pkey = _load_pkey(SSH_KEY, SSH_PASS)
ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(hostname=MASTER, username='opc', pkey=pkey, timeout=30)
sftp = ssh.open_sftp()
try:
    try: sftp.stat(LOCUST_WORKDIR)
    except FileNotFoundError: sftp.mkdir(LOCUST_WORKDIR)
finally:
    sftp.close(); ssh.close()

# Workers per host

def _detect_nproc(host: str) -> int:
    out, err, rc = ssh_exec(host, command=r"bash -lc 'nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1'", timeout=10)
    try: return max(1, int((out or '1').strip()))
    except Exception: return 1

def _desired_workers(host: str) -> int:
    if isinstance(WORKERS_PER_HOST, int): base = int(WORKERS_PER_HOST)
    elif isinstance(WORKERS_PER_HOST, str) and WORKERS_PER_HOST.lower() == 'fixed': base = MIN_WORKERS_PER_HOST
    elif isinstance(WORKERS_PER_HOST, str) and WORKERS_PER_HOST.lower() == 'auto': base = max(1, _detect_nproc(host) - int(CPU_RESERVE))
    else: base = max(1, _detect_nproc(host) - int(CPU_RESERVE))
    return max(int(MIN_WORKERS_PER_HOST), min(int(MAX_WORKERS_PER_HOST), int(base)))

EXPECTED_WORKERS = 0
per_host_workers = {}
for host in GEN_IPS:
    n = _desired_workers(host)
    per_host_workers[host] = n
    EXPECTED_WORKERS += n
print(f"workers_per_host={per_host_workers} | total={EXPECTED_WORKERS}")

# Build DEFAULT_HOST_URL for UI display and LOCUST_TARGETS for Multi
DEFAULT_HOST_URL = ""
LOCUST_TARGETS = ""

def _bracket_if_ipv6(host: str) -> str:
    return (f"[{host}]" if ":" in (host or "") and not (host.startswith("[") and host.endswith("]")) else host)

if LB_TOPOLOGY == "single" and TARGET_HOST:
    DEFAULT_HOST_URL = f"https://{TARGET_HOST}"
else:
    # Prefer first IPv6 VIP from LB_IPS_FLAT; else first VIP
    v6 = [ip for ip in (LB_IPS_FLAT or []) if ":" in ip]
    first = v6[0] if v6 else ((LB_IPS_FLAT or [""])[0])
    if first:
        DEFAULT_HOST_URL = f"https://{_bracket_if_ipv6(first)}"
    # LOCUST_TARGETS = all VIPs (dedup, IPv6 bracketed)
    if LB_IPS_FLAT:
        seen = set(); uniq = []
        for ip in LB_IPS_FLAT:
            if ip in seen: continue
            seen.add(ip); uniq.append(ip)
        LOCUST_TARGETS = ",".join([f"https://{_bracket_if_ipv6(ip)}" for ip in uniq])

# Build and upload run_ui.sh
ui_runner = rf"""#!/usr/bin/env bash
set -euo pipefail
export LOCUST_MODE={TEST_MODE}
export LOCUST_HEALTH_PATH='{HEALTH_ENDPOINT_PATH}'
export LOCUST_THROUGHPUT_PATH='{THROUGHPUT_ENDPOINT_PATH}'
export LOCUST_VERIFY_TLS={'true' if LOCUST_VERIFY_TLS else 'false'}
export LOCUST_CONNECT_TIMEOUT_S={LOCUST_CONNECT_MS/1000.0}
export LOCUST_READ_TIMEOUT_S={LOCUST_READ_MS/1000.0}
export LOCUST_WAIT_TIME_S={LOCUST_WAIT_TIME_SEC}
export PATH=$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH
{'export LOCUST_TARGETS=' + shlex.quote(LOCUST_TARGETS) if LOCUST_TARGETS else 'unset LOCUST_TARGETS'}
export LOCUST_NAME_BY_VIP={'true' if bool(LOCUST_TARGETS) else 'false'}
{'export LOCUST_DEFAULT_HOST=' + shlex.quote(DEFAULT_HOST_URL) if DEFAULT_HOST_URL else 'unset LOCUST_DEFAULT_HOST'}
cd {LOCUST_WORKDIR}
python3 -m locust -f locustfile.py --master --master-bind-host {MASTER_BIND_HOST} \
  --web-host 0.0.0.0 --web-port {UI_WEB_PORT} \
  {'--host ' + shlex.quote(DEFAULT_HOST_URL) if DEFAULT_HOST_URL else ''} \
  > {LOCUST_WORKDIR}/ui_master.log 2>&1
"""

ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(hostname=MASTER, username='opc', pkey=pkey, timeout=30)
sftp = ssh.open_sftp()
try:
    with sftp.file(f"{LOCUST_WORKDIR}/run_ui.sh", "w") as f:
        f.write(ui_runner)
    sftp.chmod(f"{LOCUST_WORKDIR}/run_ui.sh", 0o755)
finally:
    sftp.close(); ssh.close()
print("run_ui.sh uploaded")

# Clean stale workers and UI tmux
for host in GEN_IPS:
    kill_cmd = r"""bash -lc 'pkill -f "python3 -m locust .*--worker" 2>/dev/null || true; pkill -f "locust .*--worker" 2>/dev/null || true'"""
    ssh_exec(host, command=kill_cmd, timeout=10)
ssh_exec(MASTER, command="bash -lc 'tmux has-session -t locust_ui_master 2>/dev/null && tmux kill-session -t locust_ui_master || true'", timeout=10)

# Launch UI master via tmux
ssh_exec(MASTER, command=f"bash -lc {shlex.quote(f'tmux new -d -s locust_ui_master {LOCUST_WORKDIR}/run_ui.sh')}", timeout=20)

# Spawn workers on all generators
print("\nSpawning workers on all generators ...")
for host in GEN_IPS:
    n = per_host_workers[host]
    start_workers_content = (
        f"#!/usr/bin/env bash\n"
        f"set -euo pipefail\n"
        f"export LOCUST_MODE={TEST_MODE}\n"
        f"export LOCUST_HEALTH_PATH='{HEALTH_ENDPOINT_PATH}'\n"
        f"export LOCUST_THROUGHPUT_PATH='{THROUGHPUT_ENDPOINT_PATH}'\n"
        f"export LOCUST_VERIFY_TLS={'true' if LOCUST_VERIFY_TLS else 'false'}\n"
        f"export LOCUST_CONNECT_TIMEOUT_S={LOCUST_CONNECT_MS/1000.0}\n"
        f"export LOCUST_READ_TIMEOUT_S={LOCUST_READ_MS/1000.0}\n"
        f"export LOCUST_WAIT_TIME_S={LOCUST_WAIT_TIME_SEC}\n"
        f"export PATH=$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH\n"
        + (f"export LOCUST_TARGETS={shlex.quote(LOCUST_TARGETS)}\nexport LOCUST_NAME_BY_VIP=true\n" if LOCUST_TARGETS else "unset LOCUST_TARGETS\nexport LOCUST_NAME_BY_VIP=false\n") +
        (f"export LOCUST_DEFAULT_HOST={shlex.quote(DEFAULT_HOST_URL)}\n" if DEFAULT_HOST_URL else "unset LOCUST_DEFAULT_HOST\n") +
        f"cd {LOCUST_WORKDIR}\n"
        f"echo 'Spawning {n} worker(s) to master {MASTER_ADDR_FOR_WORKERS} at ' $(date -u)\n"
        f"for i in $(seq 1 {n}); do nohup python3 -m locust -f locustfile.py --worker --master-host {MASTER_ADDR_FOR_WORKERS} > locust-worker-$i.log 2>&1 & done\n"
        f"echo WORKERS_STARTED $(date -u)\n"
    )
    ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username='opc', pkey=pkey, timeout=30)
    try:
        sftp = ssh.open_sftp()
        try: sftp.stat(LOCUST_WORKDIR)
        except FileNotFoundError: sftp.mkdir(LOCUST_WORKDIR)
        with sftp.file(f"{LOCUST_WORKDIR}/start_workers.sh", "w") as f:
            f.write(start_workers_content)
        sftp.chmod(f"{LOCUST_WORKDIR}/start_workers.sh", 0o755)
        sftp.close()
        ssh.exec_command(f"bash -lc 'nohup {shlex.quote(LOCUST_WORKDIR)}/start_workers.sh > {shlex.quote(LOCUST_WORKDIR)}/start_workers.out 2>&1 &'", timeout=10)
    finally:
        ssh.close()
    print(f"{host}: requested {n} worker(s)")

# Verify worker processes
time.sleep(5)
print("\nWorker process counts (approx):")
for host in GEN_IPS:
    cmd = r"""bash -lc "ps -eo cmd | grep -E 'python3 -m locust .*--worker( |$)|locust .*--worker( |$)' | grep -v grep | wc -l" """
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try: cnt = int((out or '0').strip() or 0)
    except Exception: cnt = 0
    print(f"{host}: workers≈{cnt} (rc={rc})")

# UI access instructions
print("\nUI access (SSH tunnel recommended):")
print(f"  IPv6: ssh -6 -i {SSH_KEY} -o ExitOnForwardFailure=yes -N -L \"[::1]:8088:127.0.0.1:{UI_WEB_PORT}\" opc@{MASTER}")
print("       Then open: http://[::1]:8088")
try:
    master_public_ipv4 = (GEN_IPS_V4 or [None])[0]
except Exception:
    master_public_ipv4 = None
if master_public_ipv4:
    print(f"  IPv4: ssh -i {SSH_KEY} -o ExitOnForwardFailure=yes -N -L 8089:127.0.0.1:{UI_WEB_PORT} opc@{master_public_ipv4}")
    print("       Then open: http://127.0.0.1:8089")

print("")
print("Important:")
print("  • Open the UI with http:// (not https://). The UI is HTTP-only.")
print("  • Data plane remains IPv6 VIP → TLS offload → IPv4 backends. Fan-out is via LOCUST_TARGETS.")
print("")
print("In the UI form:")
print("  - Multi: leave Host empty or keep the default shown; requests will shard across LOCUST_TARGETS (all VIPs).")
print("  - Single: Host is already provided.")
print("")
print("When finished:")
print("  • Run Cell 13 to stop all components (UI + workers).")
print("  • Run Cell 14 to teardown infrastructure (destroy resources).")
print("")
print("Cell 12 complete: UI/Workers launched. Use one tunnel above, open the UI, and drive your test from the browser.")


In [ ]:
# Cell 13 — Objective: Stop Locust UI and workers (graceful first; hard stop optional)

import time

# Fallback ssh_exec (if not already available from previous cells)
try:
    ssh_exec  # noqa
except NameError:
    import paramiko, os, json
    def _ssh_key_path():
        return os.path.expanduser((globals().get("SSH_PRIVATE_KEY_PATH") or os.environ.get("SSH_PRIVATE_KEY_PATH") or "~/.ssh/id_rsa"))
    SSH_KEY = _ssh_key_path()
    SSH_PASS = os.environ.get("SSH_KEY_PASSPHRASE", None)
    def _load_pkey(path, pw):
        for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
            try: return Key.from_private_key_file(path, password=pw)
            except Exception: pass
        raise paramiko.SSHException("Could not load SSH private key")
    def ssh_exec(host, user="opc", key_path=None, command="echo ok", timeout=120):
        pkey = _load_pkey(key_path or SSH_KEY, SSH_PASS)
        ssh = paramiko.SSHClient(); ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username='opc', pkey=pkey, timeout=30)
        _, stdout, stderr = ssh.exec_command(command, timeout=timeout)
        out = (stdout.read().decode('utf-8', errors='ignore') or '').strip()
        err = (stderr.read().decode('utf-8', errors='ignore') or '').strip()
        try: rc = stdout.channel.recv_exit_status()
        except Exception: rc = None
        ssh.close(); return out, err, rc

# Resolve MASTER/WORKERS if not already defined
try:
    MASTER  # noqa
    WORKERS # noqa
except NameError:
    import json, os
    try: GEN_IPS_V4 = json.loads(os.environ.get("GEN_IPS_V4_JSON", "[]") or "[]")
    except Exception: GEN_IPS_V4 = []
    try: GEN_IPS_V6 = json.loads(os.environ.get("GEN_IPS_V6_JSON", "[]") or "[]")
    except Exception: GEN_IPS_V6 = []
    GEN_IPS = GEN_IPS_V4 or GEN_IPS_V6
    MASTER = GEN_IPS[0] if GEN_IPS else None
    WORKERS = GEN_IPS[1:] if GEN_IPS and len(GEN_IPS) > 1 else []

TMUX_UI_MASTER = "locust_ui_master"
TMUX_HEADLESS_MASTER = "locust_headless"

# Utility: stop a tmux session if present

def _stop_tmux_session(host, session_name):
    cmd = rf"bash -lc 'tmux has-session -t {session_name} 2>/dev/null && tmux kill-session -t {session_name} || true'"
    ssh_exec(host, command=cmd, timeout=20)

# Worker process helpers

_WORKER_PATTERNS = [r"python3 -m locust .*--worker( |$)", r"locust .*--worker( |$)"]

def _pkill_workers(host):
    cmd = r"""bash -lc '
set +e
pkill -f "python3 -m locust .*--worker" 2>/dev/null || true
pkill -f "locust .*--worker" 2>/dev/null || true
true
'"""
    return ssh_exec(host, command=cmd, timeout=10)


def _count_workers(host):
    cmd = r"""bash -lc "ps -eo cmd | grep -E 'python3 -m locust .*--worker( |$)|locust .*--worker( |$)' | grep -v grep | wc -l" """
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try: return int((out or "0").strip())
    except Exception: return 0

# Stop helpers


def stop_ui_master():
    if MASTER:
        _stop_tmux_session(MASTER, TMUX_UI_MASTER)
    time.sleep(0.5)
    print("UI master stopped (tmux session terminated).")


def stop_headless_master():
    if MASTER:
        _stop_tmux_session(MASTER, TMUX_HEADLESS_MASTER)
    time.sleep(0.5)
    print("Headless master stopped (tmux session terminated).")


def stop_all_workers_hard(include_master: bool = True, wait_sec: int = 2):
    hosts = ([MASTER] if (include_master and MASTER) else []) + (WORKERS or [])
    if not hosts:
        print("No generator hosts to stop.")
        return
    before = {h: _count_workers(h) for h in hosts}
    for h in hosts:
        _pkill_workers(h)
        print(f"[{h}] pkill issued (workers before={before[h]})")
    time.sleep(wait_sec)
    after = {h: _count_workers(h) for h in hosts}
    for h in hosts:
        print(f"[{h}] workers remaining≈{after[h]}")
    print("Worker hard-stop sweep complete.")


def stop_everything(include_master_workers: bool = True):
    print("Stopping UI ...")
    stop_ui_master()
    print("Stopping headless master ...")
    stop_headless_master()
    print("Stopping workers (hard kill) ...")
    stop_all_workers_hard(include_master=include_master_workers)
    print("All Locust components stopped.")

print("Cell 13 complete: stop/cleanup helpers ready.")
print("Examples:")
print("  stop_ui_master()")
print("  stop_headless_master()")
print("  stop_all_workers_hard()")
print("  stop_everything()  # UI, headless master, and workers")

stop_everything()

In [ ]:
# Cell 14 — Objective: Teardown (guarded)

TEARDOWN_CONFIRM = True  # Set True to allow destroy

if TEARDOWN_CONFIRM:
    !terraform destroy -auto-approve -var-file=terraform.tfvars
    print("Cell 14 complete: All Terraform resources destroyed.")
else:
    print("Teardown guard is False. Set TEARDOWN_CONFIRM=True to destroy resources.")
    print("Cell 14 complete: No teardown executed.")
